# Sennesh et al. (2025) on dandiset 000253: V1 spectral and laminar claims

**Paper.** Sennesh E, Westerberg JA, Spencer-Smith J, Bastos AM (2025). Ubiquitous predictive
processing in the spectral domain of sensory cortex. bioRxiv,
[doi:10.1101/2025.07.31.667946](https://doi.org/10.1101/2025.07.31.667946). The claims file
(`claims/sennesh-2025.yaml`) was written from the preprint's full text (Results, legends of
Figures 1 to 5 and Supplementary Figures 1 to 4, Methods) and the code at
github.com/BastosLab/epych commit cee2e6f.

**Dandiset.** [DANDI:000253](https://dandiarchive.org/dandiset/000253/0.240923.1441), version
0.240923.1441, the most recent published version. The paper cites version 0.240503.0152. That
version cannot be used for half of the mice: in 42 of its 84 probe files (7 of the 14 mice) the
LFP timestamps run 0, 24, 48, ... (not seconds), so the LFP cannot be put on the stimulus clock;
two of those mice hold almost the same LFP as two others; and one mouse holds one LFP channel in
four. Version 0.240923.1441 replaced those 42 files and holds 14 mice with LFP timestamps in
seconds. The differences are shown in the data check below. The session files (stimulus tables,
eye tracking) are the same blobs in both versions.

**What this notebook tests.** The claims that rest on V1 alone: the trial counts, the
V1 condition spectrograms, cluster-tested contrasts and laminar panels (Figures 2 to 4, a to f),
the trough-locked CSD (Figures 2 to 4, g to i), the V1 gamma peak frequency, the theta peaks,
and the pupil (Supplementary Figure 4). The claims across all six areas are in
`areas.ipynb`.

| Claim | Tested here |
|---|---|
| trial-counts | yes |
| three-contrasts-gamma-increase | yes (V1) |
| fig2ab to fig2i, fig3ab to fig3i, fig4ab to fig4i (25 claims) | yes |
| dd-gamma-peak-frequency-v1, go-theta-peaks | yes |
| suppfig4-pupil-dd-no-difference | yes |
| claims on all six areas (10 claims) | in `areas.ipynb` |

In [ ]:
import time
import resource
import zlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import glo

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
PAPER = "sennesh-2025"
OFFS = [0, 1000, 2000, 3000, 4000]     # seed offsets (protocol: the claim's seed and four more)
started = time.time()


def seed(claim):
    return zlib.crc32(f"{PAPER}/{claim}".encode())


sess, smeta, pmeta = glo.survey()
mice = sorted(smeta)
print(f"dandiset {glo.DANDISET_ID} version {glo.DANDISET_VERSION}: {len(mice)} subjects, "
      f"{sum(len(v) for v in sess.values())} files; survey {time.time() - started:.0f} s")

## Data check

In [ ]:
subj = pd.DataFrame([{"mouse": m, **smeta[m][2], "eye_tracking": smeta[m][3] is not None,
                      "habituated": smeta[m][1].attrs["habituated"]} for m in mice])
print(subj.to_string(index=False))
print("\nsex:", subj.sex.value_counts().to_dict(), " genotype:",
      subj.genotype.str.split("/").str[0].value_counts().to_dict())
print("age range (days):", subj.age.str.strip("PD").astype(int).min(), "to", subj.age.str.strip("PD").astype(int).max())
print("habituated sequence (a = 45 deg, b = 135 deg):", subj.habituated.value_counts().to_dict())

The paper reports 16 mice (6 male, 10 female, 95 to 128 days old), of which 14 were kept. The
dandiset holds 14 subjects, 4 male and 10 female, so the two excluded mice are not deposited and
both were male if the paper's counts are right. One mouse is 88 days old, below the paper's
range of 95 to 128 days. Eight mice were habituated to "bbba" and six to "aaab", where the paper
describes two cohorts of 7. Five sessions have no eye tracking.

In [ ]:
rows = []
for m in mice:
    s = smeta[m][1]
    blk = s.groupby("kind").agg(sequences=("pattern", "size"), blocks=("block", "nunique"))
    c = glo.conditions(s)
    odd = s[(s.kind == "main") & ~s.xy.isin(["xxxy", "xxxx"])]
    rows.append({"mouse": m, **{f"{k}_seq": int(blk.loc[k, "sequences"]) for k in blk.index},
                 **{k: len(v) for k, v in c.items()}, "odd_main": len(odd),
                 "first50_end_block": int(c["lonaive"].block.max()),
                 "p4_dur_ms": f"{1000 * (s.p4_off - s.p4_on).min():.0f}-{1000 * (s.p4_off - s.p4_on).max():.0f}"})
counts = pd.DataFrame(rows).set_index("mouse")
print(counts.to_string())
print("\ntotals:", counts[glo.CONDITIONS].sum().to_dict())
first = smeta[mice[0]][1]
print("\nsequence period (s), median over sessions:", np.round(np.median([np.median(np.diff(smeta[m][1].p4_on)) for m in mice]), 3))

The stimulus table (`init_grating_presentations`) has no labels for blocks or conditions. Each
sequence is four gratings, 0.5 s on and 0.5 s apart, followed by a blank row; sequences start
every 5 s. Blocks are told apart by their patterns: the main block holds only the habituated
sequence and its global oddball, the random block every pattern, the sequence block only "xxxx"
and "yyyy". The main block starts with 25 habituated sequences only (block 1) and the global
oddballs begin in block 3, so the first 50 "xxxy" (the code's lonaive) include about 25 from
the main block proper. A few main-block sequences (`odd_main`) match neither pattern, likely
stimulus or table glitches; they are left out. Sub-621890 has a different schedule (807
main-block sequences, 100 sequence-control sequences, the others 748 and 200).

In [ ]:
pm = pd.DataFrame([{"mouse": x[0], "probe": x[1], "area": x[5], "lfp_rows": x[6][0], "lfp_channels": x[6][1],
                    "chunk": x[7], "ts0": x[8]["ts"][0], "ts1": x[8]["ts"][1], "ts_last": x[8]["ts"][2],
                    "n_area_channels": int((x[4].location.map(glo.area_of) == x[5]).sum()) if x[5] else 0}
                   for x in pmeta])
pm["rate_hz"] = 1 / (pm.ts1 - pm.ts0)
pm["duration_s"] = pm.ts_last - pm.ts0
print(pm.drop(columns=["ts1"]).to_string(index=False))
print("\nprobes per area label:", pm.area.value_counts(dropna=False).to_dict())
print("mice without an AL probe:", sorted(set(mice) - set(pm[pm.area == "AL"].mouse)))
print("mice with two probes of one area:", pm.groupby(["mouse", "area"]).size().loc[lambda s: s > 1].to_dict())

In [ ]:
# Interval tables of the first session file: is there a habituation table apart from
# init_grating_presentations, and which stimulus blocks does each table cover?
f_ = glo.open_h5(sess[mice[0]]["ogen"])
for k in f_["intervals"]:
    g = f_[f"intervals/{k}"]
    n_ = len(g["start_time"]) if "start_time" in g else None
    blk = sorted(set(np.asarray(g["stimulus_block"][:]).tolist())) if "stimulus_block" in g else None
    print(f"{mice[0]} intervals/{k}: {n_} rows; stimulus_block values {blk}; columns {sorted(g.keys())}")
s_ = smeta[mice[0]][1]
print(s_.groupby("block").agg(kind=("kind", "first"), sequences=("pattern", "size"),
                              patterns=("xy", lambda x: x.value_counts().to_dict())).to_string())

# invalid_times of every session, and whether any analysed P4 trial (epoch -0.5 to 1.0 s) overlaps one.
def invalid_of(m):
    g = glo.open_h5(sess[m]["ogen"])["intervals/invalid_times"]
    return m, g["start_time"][:], g["stop_time"][:]
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(14) as ex:
    inv = dict((m, (a, b)) for m, a, b in ex.map(invalid_of, mice))
for m in mice:
    a, b = inv[m]
    on = np.concatenate([c.p4_on.to_numpy() for c in glo.conditions(smeta[m][1]).values()])
    hit = sum(int(((on - 0.5 < y) & (on + 1.0 > x)).sum()) for x, y in zip(a, b))
    print(f"{m}: invalid_times {[(round(float(x), 1), round(float(y), 1)) for x, y in zip(a, b)]}; analysed P4 epochs overlapping: {hit}")

Every probe file holds its LFP at 1250 Hz (sampling interval 0.8 ms) in volts, not the 2.5 kHz
of the paper's Methods (the Neuropixels LFP band is acquired at 2.5 kHz and the files hold it
downsampled). Area labels come from the electrodes table: three mice have no probe labelled AL
(two have two probes labelled LM, one has two labelled RL), and one probe of sub-632487 has no
visual-cortex label at all. Every mouse has one V1 probe. The paper says six probes targeted
V1, LM, RL, AL, PM and AM in all mice.

In [ ]:
# Recordings deposited twice: correlate a 0.2 s, 8-channel LFP sample of every pair of probe files.
keys = [(x[0], x[1]) for x in pmeta]
samples = np.array([x[8]["sample"].ravel() for x in pmeta])
z = (samples - samples.mean(1, keepdims=True)) / samples.std(1, keepdims=True)
r = z @ z.T / z.shape[1]
np.fill_diagonal(r, 0)
i, j = np.nonzero(np.triu(np.abs(r) > 0.95))
print("pairs of probe files with |r| > 0.95 on the sample:", [(keys[a], keys[b], round(r[a, b], 3)) for a, b in zip(i, j)])
print("largest |r| between files of different mice:",
      round(max(abs(r[a, b]) for a in range(len(keys)) for b in range(len(keys)) if keys[a][0] != keys[b][0]), 3))
seq_sig = {m: tuple(smeta[m][1].pattern) for m in mice}
print("sessions with identical stimulus sequences:", [(a, b) for a in mice for b in mice if a < b and seq_sig[a] == seq_sig[b]])

In [ ]:
# The cited version, 0.240503.0152: the same check on its V1 files, and its LFP clocks.
old_sess, old_smeta, old_pmeta = glo.survey(version=glo.CITED_VERSION)
old_assets, new_assets = glo.assets(glo.CITED_VERSION), glo.assets()
changed = sorted(p for p in new_assets if p in old_assets and new_assets[p]["blob"] != old_assets[p]["blob"])
print(f"files that differ between {glo.CITED_VERSION} and {glo.DANDISET_VERSION}: {len(changed)} of {len(new_assets)};",
      "mice:", sorted(set(p.split('/')[0] for p in changed)))
op = pd.DataFrame([{"mouse": x[0], "probe": x[1], "area": x[5], "channels": x[6][1], "ts0": x[8]["ts"][0],
                    "ts1": x[8]["ts"][1], "ts_last": x[8]["ts"][2]} for x in old_pmeta if x[5] == "V1"])
print(op.to_string(index=False))
# Every probe file of the mice whose files changed: the first two timestamps and the last.
bad = sorted(set(p.split('/')[0] for p in changed))
oa = pd.DataFrame([{"mouse": x[0], "probe": x[1], "area": x[5], "ts0": x[8]["ts"][0], "ts1": x[8]["ts"][1],
                    "ts_last": x[8]["ts"][2]} for x in old_pmeta if x[0] in bad])
print(f"\ncited version, all {len(oa)} probe files of those {len(bad)} mice:")
print(oa.to_string(index=False))
print("files with timestamps 0, 24, ...:", int(((oa.ts0 == 0) & (oa.ts1 == 24)).sum()), "of", len(oa))
okeys = [(x[0], x[1]) for x in old_pmeta]
osamp = np.array([x[8]["sample"][:, :4].ravel() for x in old_pmeta])
oz = (osamp - osamp.mean(1, keepdims=True)) / osamp.std(1, keepdims=True)
orr = oz @ oz.T / oz.shape[1]
np.fill_diagonal(orr, 0)
i, j = np.nonzero(np.triu(orr > 0.95))
print("\ncited version, probe files of different mice with r > 0.95 (first 4 channels):")
for a, b in zip(i, j):
    if okeys[a][0] != okeys[b][0]:
        print("  ", okeys[a], okeys[b], round(orr[a, b], 4))

In the cited version, 0.240503.0152, all 42 probe files of 7 mice (printed above) have
timestamps 0, 24, 48, ... with no unit of time, so their LFP cannot be put on the
stimulus clock; in sub-649323 the probe files hold one channel in four; and the LFP of
sub-645495 is almost the same as that of sub-645322, and that of sub-645324 almost the same as
that of sub-647836 (correlations printed above), while their stimulus tables differ. The pinned
version replaced all 42 probe files of those 7 mice, and in it no two probe files of different
mice are alike (above). This attempt uses the pinned version.

### Reading V1

The V1 probe of each mouse is read whole, for its visual-cortex channels, from the HDF5 chunk
index. The time taken and the peak memory are printed, as they bound what else can be run.

In [ ]:
v1 = {x[0]: x for x in pmeta if x[5] == "V1"}
chans = {m: glo.cortical_channels(v1[m][4], "V1") for m in mice}
spans, labels, kept, raw_labels = glo.alignment(chans)
kl = [labels[k] for k in kept]
print("aligned rows:", len(labels), " kept (every fourth):", len(kept))
print("kept rows' layers:", kl)
print("kept rows (top-down aligned row numbers, every fourth from the deepest):", list(kept))
print("aligned rows labelled L1:", [j for j, l in enumerate(labels) if l == "L1"])
for k in [1, 2, 3]:
    lk = [labels[j] for j in glo.kept_rows(len(labels), k)]
    print(f"start moved up by {k} rows: {len(lk)} kept rows, L1 rows {lk.count('L1')}, first three {lk[:3]}")
print("LFP columns in local-index order (tip first), every V1 file:",
      all(bool((np.diff(v1[m][4].local_index.to_numpy()) > 0).all()) for m in mice))
print("cortical V1 channels per mouse:", {m: len(c) for m, c in chans.items()})
t = time.time()
metas = glo.chunk_indexes([(v1[m][2], v1[m][3]) for m in mice], workers=14)
metas = dict(zip(mice, metas))
print(f"chunk indexes of {len(mice)} V1 files read in {time.time() - t:.0f} s")
conds = {m: glo.conditions(smeta[m][1]) for m in mice}
passes = {}
for m in mice:
    t = time.time()
    passes[m] = glo.mouse_pass(v1[m][2], metas[m], chans[m], spans[m], labels, kept, conds[m])
    print(f"{m}: {passes[m]['n_pairs']} chunk columns read in {passes[m]['read_s']:.0f} s, pass {time.time() - t:.0f} s; "
          f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")
print(f"V1 read in {(time.time() - started) / 60:.1f} min since the start")

In [ ]:
clock = pd.DataFrame([{"mouse": m, "t0": passes[m]["t0"], "fs_fitted": passes[m]["fs"],
                       "max_residual_ms": 1000 * passes[m]["clock_resid"]} for m in mice])
print("LFP clock fitted to 12 timestamps spread over each file:")
print(clock.to_string(index=False))
# Clock check: the evoked LFP of P4 for xxxY (main block, P4 = y) against xxxX (main block,
# P4 = x). The two differ only in P4's orientation, so a true onset shows their difference
# starting after P4 onset, and not before it.
tt = glo.epoch_times()
win = [(-0.3, -0.2), (-0.2, -0.1), (-0.1, 0.0), (0.0, 0.03), (0.03, 0.06), (0.06, 0.1), (0.1, 0.2), (0.2, 0.4)]
rows = []
for m in mice:
    a = passes[m]["erp_sum"]["lo_gloexp"] / passes[m]["n"]["lo_gloexp"]
    b = passes[m]["erp_sum"]["go_gloexp"] / passes[m]["n"]["go_gloexp"]
    d = np.abs(a - b).mean(0) * 1e6
    rows.append({"mouse": m, **{f"{int(w0 * 1000)}..{int(w1 * 1000)}ms": d[(tt >= w0) & (tt < w1)].mean() for w0, w1 in win}})
ck = pd.DataFrame(rows).set_index("mouse")
print("\nMean |ERP(xxxY) - ERP(xxxX)| over channels (microvolts), by window around P4 onset:")
print(ck.round(1).to_string())
print("\nratio of 60-100 ms to -100-0 ms:", (ck["60..100ms"] / ck["-100..0ms"]).round(2).to_dict())

All V1 files have a linear clock in seconds (residuals printed). In every mouse the evoked
difference between the two P4 orientations is small before P4 onset and grows after about
30 to 60 ms, as a visual response does; so the LFP and the stimulus table share a clock in
each session, as far as this contrast shows.

## Spectrograms, aperiodic removal and cluster tests (S2 to S5)

Trials are pooled over mice as observations. FOOOF is fitted per condition and kept channel to
the spectrogram averaged over trials and time (reading a: all frequencies, 5 to 150 Hz; reading
b: 5 to 90 Hz). The fast cluster test is first checked against MNE's on a small subset.

The first time bin is centred at -10 ms, and its 200 ms window runs from -110 to +90 ms, so a
value there already holds up to 90 ms of the response. A "first" time of -10 ms in the laminar
tables means within the first 90 ms after onset, and an order between -10 and 0 ms (or any two
bins 10 ms apart) is finer than the window resolves.

In [ ]:
import mne

rng = np.random.default_rng(0)  # standard choice: synthetic check data only
a_ = rng.normal(size=(30, 8, 6, 5))
b_ = rng.normal(size=(25, 8, 6, 5))
a_[:, 2:5, 1:4, 1:3] += 1.0
thr = glo.f_threshold(30, 25)
f_m, cl_m, p_m, _ = mne.stats.spatio_temporal_cluster_test([a_, b_], threshold=thr, n_permutations=1000, tail=1,
                                                           seed=1, out_type="mask", verbose=False)
f_g, lab_g, p_g = glo.cluster_test(a_, b_, np.random.default_rng(1), threshold=thr)
print("F maps equal:", np.allclose(f_m, f_g))
print("MNE clusters (size, p):", sorted([(int(c.sum()), round(p, 3)) for c, p in zip(cl_m, p_m)], reverse=True)[:4])
print("glo clusters (size, p): ", sorted([(int((lab_g == k + 1).sum()), round(p, 3)) for k, p in enumerate(p_g)], reverse=True)[:4])

In [ ]:
freqs = glo.FOI
times = glo.TOI
print("spectrogram frequencies, label: FFT bin (Hz):", ", ".join(f"{a:.0f}: {b:.1f}" for a, b in zip(freqs, glo.FOI_HZ)))
power = {}
for c in glo.CONDITIONS:
    power[c] = np.concatenate([passes[m]["spec"][c] for m in mice])
    for m in mice:
        passes[m]["spec"][c] = None
trial_mouse = {c: np.concatenate([[m] * passes[m]["n"][c] for m in mice]) for c in glo.CONDITIONS}
print({c: power[c].shape for c in power})
FIT = {"a": [5, 150], "b": [5, 90]}
aperiodic = {}
for r, rng_ in FIT.items():
    for c in glo.CONDITIONS:
        mean = power[c].mean((0, 1))           # freq, channel
        for k in range(mean.shape[1]):
            ap, params, fm = glo.fooof_aperiodic(freqs, mean[:, k], rng_)
            aperiodic[(r, c, k)] = (ap, params, fm.r_squared_)
fit = pd.DataFrame([{"reading": r, "condition": c, "channel": k, "offset": p[0], "exponent": p[1], "r2": q}
                    for (r, c, k), (ap, p, q) in aperiodic.items()])
print(fit.groupby(["reading", "condition"])[["offset", "exponent", "r2"]].agg(["min", "max"]).round(3).to_string())


def db(c, r):
    ap = np.stack([aperiodic[(r, c, k)][0] for k in range(power[c].shape[-1])], -1)   # freq, channel
    return (10 * (np.log10(power[c]) - ap[None, None])).astype(np.float32)

In [ ]:
results = {}
SEEDS = {"DD": "fig2c-dd-v1-contrast", "GO": "fig3c-go-v1-contrast", "SSA": "fig4c-ssa-v1-contrast"}
F5SEED = "fig5-l23-gamma-by-area"     # the areas notebook's seed, for the comparison with its light pass
for r in FIT:
    cond_mean = {}
    for name, (ca, cb) in glo.CONTRASTS.items():
        da, dbb = db(ca, r), db(cb, r)
        cond_mean[ca], cond_mean[cb] = da.mean(0), dbb.mean(0)
        for off in OFFS + ["fig5"]:
            t = time.time()
            s_ = seed(F5SEED) if off == "fig5" else seed(SEEDS[name]) + off
            f_obs, lab, pv = glo.cluster_test(da, dbb, np.random.default_rng(s_))
            sig = np.isin(lab, 1 + np.flatnonzero(pv < 0.01))
            diff = cond_mean[ca] - cond_mean[cb]
            results[(r, name, off)] = {"mask": sig, "diff": diff, "masked": np.where(sig, diff, 0.0),
                                       "n_clusters": len(pv), "n_sig": int((pv < 0.01).sum()),
                                       "p_sorted": np.sort(pv)[:5]}
            if off == 0:
                print(f"reading {r} {name}: {len(da)} vs {len(dbb)} trials, {len(pv)} clusters, "
                      f"{int((pv < 0.01).sum())} with p < 0.01, smallest p {np.round(np.sort(pv)[:4], 3)}; "
                      f"{time.time() - t:.0f} s")
        del da, dbb
    # S6, two readings: the trial mean of per-trial dB (cond_mean), and dB of the trial-mean power
    # over the aperiodic fit (cond_lin, the code's oscillatory() -> erp() -> decibels()); both
    # then averaged over channels.
    results[(r, "cond_mean")] = {c: cond_mean[c].mean(-1) for c in glo.CONDITIONS}   # time, freq
    results[(r, "cond_lin")] = {}
    for c in glo.CONDITIONS:
        ap = np.stack([aperiodic[(r, c, k)][0] for k in range(power[c].shape[-1])], -1)
        results[(r, "cond_lin")][c] = (10 * (np.log10(power[c].mean(0)) - ap[None])).mean(-1)
    del cond_mean
n_trials = {c: int(power[c].shape[0]) for c in glo.CONDITIONS}
del power
print(f"cluster tests done, {(time.time() - started) / 60:.1f} min since the start; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

In [ ]:
L = np.array(kl)
GROUPS = {"L1": L == "L1", "L2/3": L == "L2/3", "L4": L == "L4", "L5/6": L == "L5/6"}


def fbins(lo, hi):
    """Frequency bins from lo to hi inclusive (claims' ranges such as 40-55 Hz)."""
    return (freqs >= lo) & (freqs <= hi)


def band(name):
    return glo.band_bins(freqs, glo.BANDS[name])


def tsel(lo, hi):
    return (times >= lo) & (times <= hi)


def panel_c(r, name, off=0):
    """Masked contrast averaged over channels: time, freq."""
    return results[(r, name, off)]["masked"].mean(-1)


def laminar(r, name, bname, off=0):
    """Masked contrast averaged over a band's bins: time, channel."""
    return results[(r, name, off)]["masked"][:, band(bname), :].mean(1)


def show(arr, title, ylab, yt=None):
    fig, ax = plt.subplots(figsize=(5, 3))
    v = np.nanmax(np.abs(arr)) or 1
    ax.imshow(arr.T, aspect="auto", origin="lower" if yt is None else "upper", cmap="RdBu_r", vmin=-v, vmax=v,
              extent=[times[0] * 1000, times[-1] * 1000, 0, arr.shape[1]])
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("ms from P4 onset")
    ax.set_ylabel(ylab)
    plt.show()
    print(f"{title}: range {np.nanmin(arr):.2f} to {np.nanmax(arr):.2f} dB (colour scale symmetric, nothing clipped)")

### Variants: the median filter and where the downsampling starts

Two checks asked for in verification. (1) The areas notebook reads only the kept rows, without
the S1 median filter. Here the V1 cluster tests are repeated on the kept rows without the filter
(`nofilt`), under the areas notebook's seed, and its Figure 5 values (L2/3 gamma) are set beside
the filtered ones under the same seed. (2) The kept rows follow the code: every fourth row from
the deepest, the topmost common row being left out by the code's crop (see `glo.alignment`).
The tests are repeated with the start moved up by 1 and by 3 rows (`shift1`, `shift3`), under
each contrast's own seed. The printout above gives which rows are L1: the main analysis and
`shift3` each keep one L1 row, a different one, and `shift1` keeps none (nor would a shift of 2,
not run, to stay within memory). The laminar claims that rest on L1 print these variants.

In [ ]:
VLAB = {"nofilt": np.array(kl), **{f"shift{k}": np.array([labels[j] for j in glo.kept_rows(len(labels), k)]) for k in glo.KEPT_SHIFTS}}
variant = {}
for vname in VLAB:
    t = time.time()
    vpow = {}
    for c in glo.CONDITIONS:
        vpow[c] = glo.variant_power([passes[m]["variants"][vname][c] for m in mice])
        for m in mice:
            passes[m]["variants"][vname][c] = None
    for r, rng_ in FIT.items():
        ap = {}
        for c in glo.CONDITIONS:
            mean = vpow[c].mean((0, 1))
            ap[c] = np.stack([glo.fooof_aperiodic(freqs, mean[:, k], rng_)[0] for k in range(mean.shape[1])], -1)
        for name, (ca, cb) in glo.CONTRASTS.items():
            da = (10 * (np.log10(vpow[ca]) - ap[ca][None, None])).astype(np.float32)
            dbb = (10 * (np.log10(vpow[cb]) - ap[cb][None, None])).astype(np.float32)
            s_ = seed(F5SEED) if vname == "nofilt" else seed(SEEDS[name])
            f_obs, lab, pv = glo.cluster_test(da, dbb, np.random.default_rng(s_))
            sig = np.isin(lab, 1 + np.flatnonzero(pv < 0.01))
            variant[(vname, r, name)] = {"mask": sig, "masked": np.where(sig, da.mean(0) - dbb.mean(0), 0.0),
                                         "n_sig": int((pv < 0.01).sum())}
            del da, dbb
    del vpow
    print(f"variant {vname}: {len(VLAB[vname])} rows; tests done in {time.time() - t:.0f} s; "
          f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")


def l23_v1(masked, mask, lab, bname):
    """Figure 5 value as the areas notebook computes it: L2/3 rows, band bins, mean over bins,
    time and rows of the masked contrast; and the number of significant L2/3 band points."""
    bb, rows_ = band(bname), lab == "L2/3"
    return float(masked[:, bb][:, :, rows_].mean(1).mean(0).mean()), int(mask[:, bb][:, :, rows_].sum())


rows = []
for r in FIT:
    for name in glo.CONTRASTS:
        a_, b_ = results[(r, name, "fig5")], variant[("nofilt", r, name)]
        for bname in ["gamma", "gamma_code"]:
            v0, n0 = l23_v1(a_["masked"], a_["mask"], L, bname)
            v1_, n1_ = l23_v1(b_["masked"], b_["mask"], VLAB["nofilt"], bname)
            rows.append({"S3": r, "contrast": name, "band": bname, "filtered_value": v0, "filtered_sig_points": n0,
                         "nofilt_value": v1_, "nofilt_sig_points": n1_})
        both, either = (a_["mask"] & b_["mask"]).sum(), (a_["mask"] | b_["mask"]).sum()
        print(f"S3 {r} {name}: significant points filtered {int(a_['mask'].sum())}, without filter {int(b_['mask'].sum())}, "
              f"shared {int(both)} of {int(either)} in either; significant clusters {a_['n_sig']} and {b_['n_sig']}")
print("V1 L2/3 gamma (Figure 5 value, whole window), with and without the S1 median filter, areas notebook's seed:")
print(pd.DataFrame(rows).round(3).to_string(index=False))


def layer_table(name, bname):
    """Per start of the kept rows and S3 reading: per layer group, the largest increase and decrease
    of the masked band contrast and the first time bin with a positive and a negative value."""
    out = []
    for vname, lab_ in [("shift0", L)] + [(f"shift{k}", VLAB[f"shift{k}"]) for k in glo.KEPT_SHIFTS]:
        for r in FIT:
            lam = laminar(r, name, bname) if vname == "shift0" else variant[(vname, r, name)]["masked"][:, band(bname), :].mean(1)
            for g in ["L1", "L2/3", "L4", "L5/6"]:
                x = lam[:, lab_ == g]
                if x.shape[1] == 0:       # no kept row in this group
                    out.append({"start_row": vname, "reading": r, "group": g, "rows": 0})
                    continue
                tp, tn = np.flatnonzero((x > 0).any(1)), np.flatnonzero((x < 0).any(1))
                out.append({"start_row": vname, "reading": r, "group": g, "rows": int((lab_ == g).sum()),
                            "max": x.max(), "min": x.min(), "pos_points": int((x > 0).sum()), "neg_points": int((x < 0).sum()),
                            "first_pos_ms": 1000 * times[tp[0]] if len(tp) else np.nan,
                            "first_neg_ms": 1000 * times[tn[0]] if len(tn) else np.nan})
    df = pd.DataFrame(out)
    print(f"{name} {bname} laminar panel by start of the kept rows (shift0 is the main analysis):")
    print(df.round(3).to_string(index=False))
    return df

## trial-counts

> Pooled over the 14 mice, the analysed P4 trials number 541 for "XXXY" (random control) and 700
> for "xxxy" (habituation), 2041 for "xxxX" and 1355 for "xxxx", and 8462 for "xxxY" and 1352 for
> "yyyy".

In [ ]:
exp = {"lo_rndctl": 541, "lonaive": 700, "go_gloexp": 2041, "go_seqctl": 1355, "lo_gloexp": 8462, "igo_seqctl": 1352}
tc = pd.DataFrame({"paper": exp, "dandiset": {c: n_trials[c] for c in exp}})
tc["difference"] = tc.dandiset - tc.paper
tc["percent"] = (100 * tc.difference / tc.paper).round(1)
print(tc.to_string())
print("\nper mouse:")
print(counts[list(exp)].to_string())
alt = counts.drop(index="sub-621890", errors="ignore")[list(exp)].sum()
print("\nwithout sub-621890 (other schedule):", alt.to_dict())

**Result: partially reproduced.** The dandiset's stimulus tables give 538, 700, 2046, 1351, 8465
and 1351 trials against the paper's 541, 700, 2041, 1355, 8462 and 1352: the habituation count
matches and the other five differ by 1 to 5 trials (at most 0.6%), inside the criterion's 5% bound.
The table holds 25 pure "xxxy" sequences in its first block (printed above for sub-621890), so the
50 habituation trials are those 25 and the first 25 of the main block; no other interval table
holds a habituation block (`invalid_times` holds one empty row per session).

## three-contrasts-gamma-increase

> Each of the three oddball contrasts (deviance detection, global oddball, stimulus-specific
> adaptation) shows a significant increase in gamma power between 30 and 60 Hz for the oddball
> over its control.

In [ ]:
rows = []
for r in FIT:
    for name in glo.CONTRASTS:
        for off in OFFS:
            mk = results[(r, name, off)]["masked"][:, fbins(30, 60), :]
            rows.append({"reading": r, "contrast": name, "seed_offset": off, "pos_points_30_60": int((mk > 0).sum()),
                         "max_dB": round(float(mk.max()), 3), "neg_points_30_60": int((mk < 0).sum())})
g3 = pd.DataFrame(rows)
print(g3.to_string(index=False))

**Result: reproduced.** In V1 each contrast has significant positive values between 30 and 60
Hz under both S3 readings: DD 2277, GO 1554 and SSA 2060 masked points (reading a; 1832, 460 and
1748 under reading b), largest 5.59, 1.37 and 2.82 dB. The counts are the same under all five seeds.

## fig2ab-dd-condition-spectra

> In V1, the response to "XXXY" (random) shows an increase then a decrease of alpha/beta power after
> onset and a long-lasting 2.5-5 dB gamma (40-55 Hz) increase; the response to habituated "xxxy"
> shows a fast increase of alpha/beta power above 5 dB that returns to baseline, followed by a
> long-lasting 2.5-5 dB gamma increase.

In [ ]:
S6 = {"lin": "dB of trial-mean power (code)", "dbmean": "trial mean of dB"}


def cond_panel(r, c, s6="lin"):
    """Condition panel (time, freq), dB over the aperiodic fit, channel mean; s6 picks the reading."""
    return results[(r, "cond_lin" if s6 == "lin" else "cond_mean")][c]


def bandmean(arr, lo, hi):
    return arr[:, fbins(lo, hi)].mean(1)


AB = (10, 25)        # alpha/beta bins 10-30 Hz by S5's rule: 10, 15, 20, 25 Hz


def summary_cond(r, c, s6):
    p = cond_panel(r, c, s6)
    ab = p[:, fbins(*AB)].mean(1)
    g = p[:, fbins(40, 55)]
    return {"S3": r, "S6": s6, "condition": c, "ab_at_-10ms": ab[0],
            "ab_max_0_150": ab[tsel(0, 0.15)].max(), "ab_t_max_ms": 1000 * times[tsel(0, 0.5)][ab[tsel(0, 0.5)].argmax()],
            "ab_min_150_500": ab[tsel(0.15, 0.5)].min(),
            "g4055_max_0_500": g[tsel(0, 0.5)].max(), "g4055_max_250_500": g[tsel(0.25, 0.5)].max(),
            "g4055_max_150_500": g[tsel(0.15, 0.5)].max(), "g4055_mean_150_500": g[tsel(0.15, 0.5)].mean(),
            "panel_min": p.min(), "panel_max": p.max()}


for r in FIT:
    for c in ["lo_rndctl", "lonaive"]:
        show(cond_panel(r, c), f"reading {r}: {c} dB of trial-mean power over aperiodic, channel mean", "frequency bin (5-150 Hz)")
print(pd.DataFrame([summary_cond(r, c, s6) for r in FIT for s6 in S6 for c in ["lo_rndctl", "lonaive"]]).round(2).to_string(index=False))

In [ ]:
rows = []
for r in FIT:
    for s6 in S6:
        x = cond_panel(r, "lo_rndctl", s6)
        n = cond_panel(r, "lonaive", s6)
        abx, abn = bandmean(x, *AB), bandmean(n, *AB)
        gx = x[:, fbins(40, 55)]
        tpk = np.flatnonzero(tsel(0, 0.5))[abx[tsel(0, 0.5)].argmax()]
        rise = abx[tpk] - abx[0]
        fall = abx[tpk] - abx[tpk:][times[tpk:] <= 0.5].min()
        rows.append({"S3": r, "S6": s6,
                     "XXXY_g4055_max": gx[tsel(0, 0.5)].max(),
                     "XXXY_g_in_range": 2.25 <= gx[tsel(0, 0.5)].max() <= 5.5,
                     "XXXY_g4055_max_after_250": gx[tsel(0.25, 0.5)].max(),
                     "XXXY_g_lasts_past_250 (max after 250 >= 2.25)": gx[tsel(0.25, 0.5)].max() >= 2.25,
                     "xxxy_ab_at_-10ms": abn[0],
                     "xxxy_ab_max_0_150": abn[tsel(0, 0.15)].max(),
                     "xxxy_ab_ge_4.5": abn[tsel(0, 0.15)].max() >= 4.5,
                     "xxxy_ab_min_150_500": abn[tsel(0.15, 0.5)].min(),
                     "xxxy_ab_back_below_1": abn[tsel(0.15, 0.5)].min() < 1,
                     "XXXY_ab_at_-10ms": abx[0], "XXXY_ab_peak": abx[tpk], "XXXY_ab_peak_ms": 1000 * times[tpk],
                     "XXXY_ab_rise_from_-10ms": rise, "XXXY_ab_fall_after_peak": fall,
                     "XXXY_ab_rise_then_fall (each > 1 dB)": rise > 1 and fall > 1})
f2ab = pd.DataFrame(rows)
print(f2ab.round(2).T.to_string())

**Result: partially reproduced, resting on an assumption.** Read as the code draws the panels
(dB of the trial-mean power over the aperiodic fit), the shapes hold: the "XXXY" 40-55 Hz maximum
lasts past 250 ms (6.27 dB in S3 reading a, 5.41 dB in reading b, the same after 250 ms); the
"xxxy" alpha/beta rises from -2.54 dB at -10 ms to 2.36 dB at 80 ms and falls back below 1 dB (down
to -1.36 dB after 150 ms); "XXXY" alpha/beta rises 3.98 dB from its -10 ms level to a peak at 250
ms and falls 2.76 dB after it. The values do not all hold: the "xxxy" alpha/beta maximum is 2.36
and 2.12 dB against at least 4.5, and the "XXXY" gamma maximum is in range only under reading b.
Taken as the trial mean of per-trial dB values, the printed values lie 3.2 to 4.8 dB lower: the
"XXXY" gamma maximum (1.58 and 0.72 dB) is reached after 250 ms, so the gamma lasts, but it is
below the 2.25 dB lower bound (the column "lasts past 250" tests that level), and the "xxxy"
alpha/beta maximum stays below 0 dB. The shapes hold and the values do not, which is partially
reproduced under that reading too.

## fig2c-dd-v1-contrast

> In V1, deviance detection ("XXXY" against "xxxy") gives a significant decrease of about 1 dB in
> alpha/beta power after onset and a 1-2 dB increase in gamma power, the gamma starting near 40-65 Hz and
> descending toward 35-55 Hz over time.

In [ ]:
G = band("gamma")
ABb = band("alphabeta")


def contrast_summary(r, name, off=0):
    p = panel_c(r, name, off)
    post = tsel(0, 0.51)
    g = p[post][:, G]
    a = p[post][:, ABb]
    gm = np.where(g > 0, g, 0)
    early, late = p[tsel(0, 0.15)][:, G], p[tsel(0.3, 0.5)][:, G]
    return {"reading": r, "seed_offset": off, "gamma_max": g.max(), "gamma_pos_points": int((g > 0).sum()),
            "ab_min": a.min(), "ab_neg_points": int((a < 0).sum()),
            "gamma_peak_freq_early": freqs[G][early.max(0).argmax()] if early.max() > 0 else np.nan,
            "gamma_peak_freq_late": freqs[G][late.max(0).argmax()] if late.max() > 0 else np.nan}


for r in FIT:
    show(panel_c(r, "DD"), f"reading {r}: DD masked contrast, channel mean", "frequency bin (5-150 Hz)")
    show(results[(r, "DD", 0)]["diff"].mean(-1), f"reading {r}: DD unmasked contrast, channel mean", "frequency bin")
f2c = pd.DataFrame([contrast_summary(r, "DD", off) for r in FIT for off in OFFS])
f2c["gamma_in_0.9_2.2"] = f2c.gamma_max.between(0.9, 2.2)
f2c["ab_in_-2.2_-0.9"] = f2c.ab_min.between(-2.2, -0.9)
print(f2c.round(3).to_string(index=False))

**Result: partially reproduced.** Both clusters are significant: the largest positive gamma value
is 2.66 dB (reading a) and 2.46 dB (reading b), above the 0.9-2.2 dB range, and the largest
alpha/beta decrease is -1.48 and -1.57 dB, in range. The largest gamma value is at 45 Hz both
early and late, so the descent from 40-65 to 35-55 Hz is not seen. The same under the five seeds.

## fig2d-dd-gamma-laminar

> In V1 the deviance-induced gamma increase is strongest in superficial layers 1 to 4 (about 2 dB;
> the legend places it earliest and strongest at the L1 and L2/3 boundary), while after about
> 100 ms a weak (under 1 dB) loss of gamma power appears in L5/6.

In [ ]:
def lam_summary(r, name, bname, off=0):
    lam = laminar(r, name, bname, off)          # time, channel
    pos = lam.max()
    k = np.unravel_index(lam.argmax(), lam.shape)[1]
    sup = np.isin(L, ["L1", "L2/3", "L4"])
    late = lam[times > 0.1][:, GROUPS["L5/6"]]
    return {"reading": r, "band": bname, "seed_offset": off, "max": pos, "max_layer": L[k],
            "sup_mean_pos": np.clip(lam[:, sup], 0, None).mean(), "deep_mean_pos": np.clip(lam[:, GROUPS["L5/6"]], 0, None).mean(),
            "L56_neg_after_100_min": late.min(), "L56_neg_points_after_100": int((late < 0).sum())}


for r in FIT:
    for bname in ["gamma", "gamma_code"]:
        show(laminar(r, "DD", bname), f"reading {r}: DD laminar {bname}", "kept channel (top = surface)", yt=True)
print("kept channel layers, top to bottom:", list(L))

In [ ]:
f2d = pd.DataFrame([lam_summary(r, "DD", b, off) for r in FIT for b in ["gamma", "gamma_code"] for off in OFFS])
f2d["max_superficial"] = f2d.max_layer.isin(["L1", "L2/3", "L4"])
f2d["max_in_1.8_2.2"] = f2d["max"].between(1.8, 2.2)
f2d["sup_gt_deep"] = f2d.sup_mean_pos > f2d.deep_mean_pos
f2d["L56_loss_under_1.1"] = (f2d.L56_neg_points_after_100 > 0) & (f2d.L56_neg_after_100_min > -1.1)
print(f2d.round(3).to_string(index=False))

**Result: partially reproduced.** Under every reading (two S3 ranges, two gamma bands) the
largest gamma increase is in an L2/3 channel, the mean positive gamma over L1 to L4 (0.54 to 1.38
dB) exceeds that over L5/6 (0.06 to 0.23 dB), and L5/6 has a significant loss after 100 ms of at
most 0.31 to 0.96 dB. The largest increase is 3.02 to 3.94 dB, above 1.8-2.2 dB.

## fig2e-dd-alphabeta-laminar

> In V1, deviance detection gives a strong (about 2 dB) alpha/beta desynchronization starting in
> L2/3 and spreading into L4 and L5/6, and an alpha/beta increase of 1-1.5 dB in L1 during the stimulus.

In [ ]:
def first_time(lam, group, sign):
    x = lam[:, GROUPS[group]]
    hit = (x < 0) if sign < 0 else (x > 0)
    t = np.flatnonzero(hit.any(1))
    return 1000 * times[t[0]] if len(t) else np.nan


def ab_summary(r, name, off=0):
    lam = laminar(r, name, "alphabeta", off)
    deep = np.isin(L, ["L2/3", "L4", "L5/6"])
    return {"reading": r, "seed_offset": off, "dec_L23_L56": lam[:, deep].min(), "inc_L1": lam[:, GROUPS["L1"]].max(),
            **{f"first_neg_{g}": first_time(lam, g, -1) for g in GROUPS},
            **{f"first_pos_{g}": first_time(lam, g, +1) for g in GROUPS}}


for r in FIT:
    show(laminar(r, "DD", "alphabeta"), f"reading {r}: DD laminar alpha/beta", "kept channel (top = surface)", yt=True)

In [ ]:
f2e = pd.DataFrame([ab_summary(r, "DD", off) for r in FIT for off in OFFS])
f2e["dec_in_1.35_2.2"] = (-f2e.dec_L23_L56).between(1.35, 2.2)
f2e["L1_in_0.9_1.65"] = f2e.inc_L1.between(0.9, 1.65)
f2e["L23_first"] = (f2e["first_neg_L2/3"] <= f2e["first_neg_L4"].fillna(np.inf)) & (f2e["first_neg_L2/3"] <= f2e["first_neg_L5/6"].fillna(np.inf))
f2e["outcome"] = np.where(f2e["dec_in_1.35_2.2"] & f2e["L1_in_0.9_1.65"] & f2e.L23_first, "reproduced",
                          np.where((f2e.dec_L23_L56 < 0) & (f2e.inc_L1 > 0), "partially reproduced", "not reproduced"))
print(f2e.round(3).to_string(index=False))
print("\noutcome by S3 reading:")
print(f2e[f2e.seed_offset == 0][["reading", "outcome"]].to_string(index=False))
lt2e = layer_table("DD", "alphabeta")

**Result: partially reproduced, resting on an assumption.** The desynchronization is in range
(-2.04 dB in S3 reading a, -2.12 dB in reading b) and starts in L2/3 at 20 ms, with L4 (20 ms)
and before L5/6 (30 ms). The kept L1 row has a significant alpha/beta increase and no decrease:
0.74 dB under reading a, below the 0.9-1.65 dB range, and 1.07 dB under reading b, in range.
By reading (printed): partially reproduced under reading a, reproduced under reading b; reading
a, the code's fitting range, is adopted. The value
rests on the single kept L1 row: the start moved up by 3 rows keeps the other L1 row, where the
increase is 2.30 and 2.44 dB, and the start moved up by 1 row keeps no L1 row.

## fig2f-dd-theta

> Deviance detection gives only a weak (0.5-1 dB), late increase in theta power in V1, in L1.

In [ ]:
print("theta bins (2-10 Hz by S5's rule):", freqs[band("theta")])
rows = []
for r in FIT:
    for off in OFFS:
        lam = laminar(r, "DD", "theta", off)
        sig = lam != 0
        rows.append({"reading": r, "seed_offset": off, "n_sig": int(sig.sum()), "n_pos": int((lam > 0).sum()), "n_neg": int((lam < 0).sum()),
                     "max": lam.max(), "min": lam.min(),
                     "layers_with_sig": sorted(set(L[np.flatnonzero(sig.any(0))])),
                     "first_sig_ms": 1000 * times[np.flatnonzero(sig.any(1))[0]] if sig.any() else np.nan})
f2f = pd.DataFrame(rows)
print(f2f.round(3).to_string(index=False))
for r in FIT:
    show(laminar(r, "DD", "theta"), f"reading {r}: DD laminar theta", "kept channel (top = surface)", yt=True)
lt2f = layer_table("DD", "theta")

**Result: not reproduced.** Theta (the 5 Hz bin, the only bin of 2-10 Hz by S5's rule) has
significant values in L2/3, L4 and L5/6, of both signs (102 positive and 168 negative points in
reading a), starting at -10 ms, and reaching 2.27 dB (reading a) and 2.56 dB (reading b), above
1.1 dB. The kept L1 row has no significant theta; the other L1 row (start moved up by 3 rows) has
a late increase (1.55 and 1.84 dB, from 140 to 160 ms). The first bin at -10 ms already holds up
to 90 ms after onset.

## Trough-locked CSD (S7)

Windows are pooled over mice. The evoked potential subtracted is the trial average of each
condition over all mice, applied through the trough positions (the windows are averaged, so
the subtraction is exact). First sinks are found per layer group in the kept rows; the CSD
repeats the edge rows (Vaknin electrodes, Elephant's default), so every kept row has a value.

S7's rule (the time of the first displayed sink in each layer group) cannot order the groups
here: the display threshold, 2 SD over the number of windows, passes most points, so every group
holds a displayed sink at the first lag and the first-sink times tie. The tables then also give,
as an assumption of this attempt (not the claims file's rule), the sink onset of each group's
mean CSD: the first lag at which it turns from source to sink, counted only for a sink that is
displayed (a row of the group shows a sink) and lasts at least 1/8 of a period of the band centre
(3 or 4 samples for gamma, 8 or 9 for alpha/beta, 27 for theta). Runs of sink or source shorter
than that are first merged into their neighbours, shortest first; a group that is then a sink
throughout has its onset at the window start, and one with no turn has none (inf). In the second
verification round the verifier found that the single-sample version of this rule, used until
then, let one-sample or undisplayed sign flips decide fig3h and fig3i; the single-sample onsets
are printed beside the sustained ones, with the rule at 1/16 and 1/4 of a cycle. For each claim and reading the sign of each group's mean CSD, and where a group holds
a displayed sink, is printed at every lag (subsampled for the longer windows), so that the
patterns can be read without the images.

In [ ]:
csd_rows = L
raw_kept = np.array([raw_labels[k] for k in kept])
seg = (tt >= glo.CSD_SEGMENT[0]) & (tt < glo.CSD_SEGMENT[1])
erp = {c: sum(passes[m]["erp_sum"][c] for m in mice) / sum(passes[m]["n"][c] for m in mice) for c in glo.CONDITIONS}


MIN_CYCLE = 1 / 8                  # minimum sink and source duration, as a fraction of the band centre's period
MIN_CYCLE_VARIANTS = [1 / 16, 1 / 4]


def sustained_onset(sink, lag, k):
    """First source-to-sink turn of a boolean sink series after runs shorter than k samples are merged
    into their neighbours (shortest first); the window start when the merged series is a sink throughout."""
    s = sink.copy()
    while True:
        edges = np.flatnonzero(np.diff(s.astype(int))) + 1
        starts, ends = np.r_[0, edges], np.r_[edges, len(s)]
        short = [(e - b, b, e) for b, e in zip(starts, ends) if e - b < k]
        if len(starts) == 1 or not short:
            break
        _, b, e = min(short)
        s[b:e] = ~s[b:e]
    turn = np.flatnonzero(~s[:-1] & s[1:])
    return lag[turn[0] + 1] if len(turn) else (lag[0] if s.all() else np.inf)


def csd_result(claim, r):
    c = glo.CSD_JOBS[claim][0]
    S = sum(passes[m]["csd"][(claim, r)][0] for m in mice)
    N = sum(passes[m]["csd"][(claim, r)][1] for m in mice)
    Q = sum(passes[m]["csd"][(claim, r)][2] for m in mice)
    half = passes[mice[0]]["csd"][(claim, r)][3]
    e = erp[c][kept][:, seg]
    W = N.sum()
    tot = np.zeros((len(kept), 2 * half + 1))
    cross = np.zeros((len(kept), 2 * half + 1))
    esq = np.zeros_like(cross)
    for s in np.flatnonzero(N):
        ew = e[:, s - half:s + half + 1]
        Ss = S[s].astype(np.float64)
        tot += Ss - N[s] * ew
        ce = glo.csd(ew[None], 1.0)[0]
        cross += glo.csd(Ss[None], 1.0)[0] * ce
        esq += N[s] * ce ** 2
    mean_csd = glo.csd(tot[None], 1.0)[0] / W
    sumsq = Q - 2 * cross + esq
    sd = np.sqrt(np.maximum(sumsq / W - mean_csd ** 2, 0))
    shown = np.abs(mean_csd) > 2 * sd / W
    lag = 1000 * np.arange(-half, half + 1) / glo.FS
    first = {}
    for g in ["L1", "L2/3", "L4", "L5/6"]:
        rowsg = csd_rows == g
        hit = ((mean_csd < 0) & shown)[rowsg].any(0)
        first[g] = lag[np.flatnonzero(hit)[0]] if hit.any() else np.inf
    for g, mask in {"L5": np.char.endswith(raw_kept.astype(str), "5"), "L6": np.char.find(raw_kept.astype(str), "6") >= 0}.items():
        hit = ((mean_csd < 0) & shown)[mask].any(0)
        first[g] = lag[np.flatnonzero(hit)[0]] if hit.any() else np.inf
    # Tie-break (S7's first-sink times tie at the window start when every group holds a sink
    # there): the sink onset of each group, the first lag at which the group's mean CSD turns
    # from source (>= 0) to sink (< 0); the window start when it is a sink throughout.
    onset = {}
    gmask = {"L1": csd_rows == "L1", "L2/3": csd_rows == "L2/3", "L4": csd_rows == "L4", "L5/6": csd_rows == "L5/6",
             "L5": np.char.endswith(raw_kept.astype(str), "5"), "L6": np.char.find(raw_kept.astype(str), "6") >= 0}
    for g, mask in gmask.items():
        gm = mean_csd[mask].mean(0)
        cross = np.flatnonzero((gm[:-1] >= 0) & (gm[1:] < 0))
        onset[g] = lag[cross[0] + 1] if len(cross) else (lag[0] if (gm < 0).all() else np.inf)
    gsign = {g: mean_csd[mask].mean(0) < 0 for g, mask in gmask.items()}
    gshown = {g: ((mean_csd < 0) & shown)[mask].any(0) for g, mask in gmask.items()}
    # Sustained onset (round-2 revision of the tie-break): the same source-to-sink turn, counted
    # only for a displayed sink (the group's mean CSD is a sink and a row of the group shows a sink)
    # that lasts at least MIN_CYCLE of a period of the band centre; runs of sink or source shorter
    # than that are merged into their neighbours first. MIN_CYCLE_VARIANTS are shown beside it.
    lo, hi = glo.CSD_BANDS[r][glo.CSD_JOBS[claim][1]]
    period = glo.FS / ((lo + hi) / 2)
    sustained = {f: {g: sustained_onset(gsign[g] & gshown[g], lag, int(np.ceil(f * period)))
                     for g in gmask} for f in [MIN_CYCLE] + MIN_CYCLE_VARIANTS}
    return {"windows": int(W), "mean": mean_csd, "shown": shown, "lag": lag, "first": first, "onset": onset,
            "frac_shown": shown.mean(), "gsign": gsign, "gshown": gshown, "sustained": sustained,
            "min_samples": int(np.ceil(MIN_CYCLE * period))}


def csd_show(claim, r, res):
    fig, ax = plt.subplots(figsize=(4, 3))
    v = np.abs(res["mean"]).max()
    ax.imshow(np.where(res["shown"], res["mean"], 0), aspect="auto", cmap="RdBu", vmin=-v, vmax=v,
              extent=[res["lag"][0], res["lag"][-1], len(csd_rows), 0])
    ax.set_title(f"{claim} reading {r} (sinks red)", fontsize=9)
    ax.set_xlabel("ms from trough")
    ax.set_ylabel("kept row (top = surface)")
    plt.show()
    print(f"{claim} reading {r}: {res['windows']} windows; CSD range {res['mean'].min():.3g} to {res['mean'].max():.3g} "
          f"(V/row^2, arbitrary scale); shown fraction {res['frac_shown']:.2f}; first sink (ms from trough): "
          + ", ".join(f"{g} {v:.1f}" for g, v in res["first"].items())
          + "; sustained displayed sink onset of the group mean: "
          + ", ".join(f"{g} {v:.1f}" for g, v in res["sustained"][MIN_CYCLE].items()))


CSD = {(cl, r): csd_result(cl, r) for cl in glo.CSD_JOBS for r in ["a", "b"]}
print("CSD rows' layers (top to bottom):", list(csd_rows))
print("CSD rows' labels:", list(raw_kept))


def ff(f):  # feedforward: L4 sink before the L2/3 and L5/6 sinks
    return f["L4"] < f["L2/3"] and f["L4"] < f["L5/6"]


def fb(f):  # feedback: L1 sink before the L2/3 and L4 sinks
    return f["L1"] < f["L2/3"] and f["L1"] < f["L4"]


def csd_table(claim, rule, expected):
    out = []
    for r in ["a", "b"]:
        res = CSD[(claim, r)]
        csd_show(claim, r, res)
        tied = len(set(res["first"][g] for g in ["L1", "L2/3", "L4", "L5/6"])) == 1
        step = max(1, len(res["lag"]) // 64)
        print(f"reading {r}, by lag from {res['lag'][0]:.1f} to {res['lag'][-1]:.1f} ms, every {step * 0.8:.1f} ms: "
              "S = the group's mean CSD is a sink (< 0), . = a source; then x = a displayed sink in a row of the group")
        for g in res["gsign"]:
            print(f"  {g:5s} mean {''.join('S' if v else '.' for v in res['gsign'][g][::step])}")
            print(f"  {g:5s} shown {''.join('x' if v else '.' for v in res['gshown'][g][::step])}")
        out.append({"reading": r, "windows": res["windows"], "expected_windows": expected,
                    "within_10pct": abs(res["windows"] - expected) <= 0.1 * expected,
                    "S7_first_tied": tied, "S7_rule_holds": rule(res["first"]),
                    "single_sample_onset_rule_holds": rule(res["onset"]), "min_samples": res["min_samples"],
                    **{f"onset_{g}": v for g, v in res["sustained"][MIN_CYCLE].items()},
                    "onset_rule_holds": rule(res["sustained"][MIN_CYCLE]),
                    **{f"holds_at_1/{round(1 / f)}_cycle": rule(res["sustained"][f]) for f in MIN_CYCLE_VARIANTS}})
    print("single-sample sink onsets (round-2 tie-break, a variant now):")
    for o, r in zip(out, ["a", "b"]):
        print(f"  reading {r}: " + ", ".join(f"{g} {v:.1f}" for g, v in CSD[(claim, r)]["onset"].items()))
    print("sustained displayed sink onsets (ms from trough), adopted at 1/8 cycle, and the rule at 1/16 and 1/4 cycle:")
    print(pd.DataFrame(out).to_string(index=False))

## fig2g-dd-gamma-csd-feedforward

> Trough-locked gamma oscillations in P4 of "XXXY" in V1 show a feedforward laminar pattern, with
> sinks beginning in L4 and spreading to L2/3 and later to L5/6.

In [ ]:
csd_table("fig2g", ff, 125)

**Result: reproduced, resting on an assumption.** S7's first-sink times tie at the first lag in
both readings (every layer group holds a displayed sink there), so S7's rule cannot order them.
With the tie-break stated above (sustained, displayed sink onset of each group's mean CSD), the L4
sink starts before the L2/3 and L5/6 sinks under both bands (L4 -8.0, L2/3 -3.2, L5/6 1.6 ms in
reading a; -12.0, -4.0 and 4.8 ms in reading b), and the rule holds at 1/16 and 1/4 of a cycle and
with single-sample onsets. 120 windows (reading a, within 10% of 125) and 110 (reading b).


## fig2h-dd-alphabeta-csd-feedback

> Trough-locked alpha/beta oscillations in P4 of habituated "xxxy" in V1 show a weak feedback
> pattern, beginning in L1 and moving down through L2/3 and L4 to meet activity rising from L5.

In [ ]:
csd_table("fig2h", fb, 189)

**Result: not reproduced, resting on an assumption.** With the tie-break, the Methods' band (10-30
Hz, reading a) has no L1 sink onset in the window, and the L5/6 sink (-14.4 ms) starts before L4
(12.0 ms) and L2/3 (19.2 ms); the code's band (12-24 Hz, reading b) gives L1 first (-31.2 ms,
against L2/3 -13.6 ms, and no L4 onset). The readings disagree, and the Methods' band is adopted.
The same holds at 1/16 and 1/4 of a cycle and with single-sample onsets. S7's own rule fails in
both (tied in a; in b the first L1 sink, -31.2 ms, is after L2/3, -41.6 ms). 138 and 148 windows
against 189.


## fig2i-dd-theta-csd-mixed

> Trough-locked theta oscillations in P4 of "XXXY" in V1 begin in L1 and L6 and converge through L4
> to L2/3, followed by spread from L2/3 to L1 and L5/6, a mixed feedback and feedforward pattern.

In [ ]:
csd_table("fig2i", lambda f: min(f["L1"], f["L5/6"], f["L6"]) < f["L4"] and f["L2/3"] > f["L4"], 77)

**Result: not reproduced, resting on an assumption.** With the tie-break, under the Methods' band
(reading a) L2/3 and L4 are sinks from the window start (-124.8 ms), the L5/6 sink starts at 65.6
ms and L1 and L6 have no sink onset; under the code's band (reading b) L4 is a sink from the window
start, L5/6 and L6 start at 57.6 and 73.6 ms, and L1 and L2/3 have no onset. No L1, L5/6 or L6
sink comes before L4 in either reading, and the rule fails at 1/16 and 1/4 of a cycle and with
single-sample onsets. S7's own rule fails too. 163 and 117 windows against 77.


## dd-gamma-peak-frequency-v1

> In V1 the gamma peak frequency is higher for "XXXY" than for habituated "xxxy", by about 3 Hz
> (38 Hz, 99% CI 35.2-40.5, against 35 Hz, 99% CI 32.6-36.8).

Per mouse and condition: the trial-mean power spectrum per aligned channel, FOOOF per channel
(fixed mode, the Methods' settings), oscillatory spectrum in dB, averaged over channels.
Readings (a) window 50 ms before onset to 50 ms after offset, FOOOF 2 to 90 Hz; (b) 20 ms
margins, FOOOF 2 to 150 Hz.

In [ ]:
pf = passes[mice[0]]["psd_freqs"]
SPEC_FIT = {"a": [2, 90], "b": [2, 150]}
osc, ap_params = {}, {}
for r in ["a", "b"]:
    for c in ["lo_rndctl", "lonaive", "go_gloexp", "go_seqctl"]:
        for m in mice:
            P = passes[m]["psd"][(r, c)]          # channels, freq
            ok = pf > 0
            per = []
            for k in range(P.shape[0]):
                ap, prm, fm = glo.fooof_aperiodic(pf[ok], P[k, ok], SPEC_FIT[r])
                per.append(10 * (np.log10(P[k, ok]) - ap))
            osc[(r, c, m)] = np.mean(per, 0)
            ap_params[(r, c, m)] = glo.fooof_aperiodic(pf[ok], P[:, ok].mean(0), SPEC_FIT[r])[1]
fo = pf[pf > 0]
print("per-mouse oscillatory spectra:", len(osc), "; frequencies", fo[0], "to", fo[-1], "Hz in 2 Hz steps")

In [ ]:
gsel = (fo >= 30) & (fo <= 90)
rows, tests = [], []
for r in ["a", "b"]:
    pk = {c: np.array([fo[gsel][osc[(r, c, m)][gsel].argmax()] for m in mice]) for c in ["lo_rndctl", "lonaive"]}
    for c in pk:
        mu, se = pk[c].mean(), pk[c].std(ddof=1) / np.sqrt(len(mice))
        z99 = stats.norm.ppf(0.995)
        rows.append({"reading": r, "condition": c, "mean_peak_hz": mu, "ci99_low": mu - z99 * se, "ci99_high": mu + z99 * se,
                     "per_mouse": pk[c].tolist()})
    for off in OFFS:
        tstat, p = glo.welch_perm_test(pk["lo_rndctl"], pk["lonaive"], np.random.default_rng(seed("dd-gamma-peak-frequency-v1") + off))
        tests.append({"reading": r, "seed_offset": off, "t": tstat, "p": p})
pkv1 = pd.DataFrame(rows)
print(pkv1.drop(columns="per_mouse").round(2).to_string(index=False))
print(pkv1[["reading", "condition", "per_mouse"]].to_string(index=False))
print(pd.DataFrame(tests).round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(5, 3))
for c, col in [("lo_rndctl", "C3"), ("lonaive", "C0")]:
    y = np.array([osc[("a", c, m)] for m in mice])
    ax.plot(fo, y.mean(0), col, label=c)
    ax.fill_between(fo, y.mean(0) - 2 * y.std(0, ddof=1) / np.sqrt(len(y)), y.mean(0) + 2 * y.std(0, ddof=1) / np.sqrt(len(y)), color=col, alpha=0.2)
ax.set_xlim(2, 100)
ax.set_xlabel("Hz")
ax.set_ylabel("dB over aperiodic")
ax.legend()
plt.show()
print("plotted range (reading a, mean over mice):", round(float(min(np.array([osc[('a', c, m)] for m in mice]).mean(0)[fo <= 100].min() for c in ['lo_rndctl', 'lonaive'])), 2),
      "to", round(float(max(np.array([osc[('a', c, m)] for m in mice]).mean(0)[fo <= 100].max() for c in ['lo_rndctl', 'lonaive'])), 2), "dB")

**Result: reproduced.** The mean gamma peak is 37.57 Hz (99% interval 34.92 to 40.23) for "XXXY"
and 33.71 Hz (31.37 to 36.06) for "xxxy", both within tolerance; the permutation Welch test gives
p = 0.018 (0.012 to 0.026 over the five seeds). Both readings give the same per-mouse peaks.

## go-theta-peaks

> Oscillatory spectra with finer frequency resolution have peaks at 5-9 Hz, within the theta band,
> supporting the reading of the global oddball's low-frequency increase as theta.

In [ ]:
th = (fo >= 4.5) & (fo <= 9.9)
rows = []
for r in ["a", "b"]:
    for c in ["go_gloexp", "go_seqctl"]:
        y = np.array([osc[(r, c, m)] for m in mice]).mean(0)
        locmax = [f for i, f in enumerate(fo) if th[i] and 0 < i < len(fo) - 1 and y[i] > y[i - 1] and y[i] > y[i + 1] and y[i] > 0]
        rows.append({"reading": r, "condition": c, "spectrum_2_to_12Hz": dict(zip(fo[fo <= 12], y[fo <= 12].round(2))),
                     "local_max_above_0_in_4.5_9.9": locmax})
print(pd.DataFrame(rows).to_string(index=False))
centres = []
for c in ["go_gloexp", "go_seqctl"]:
    for m in mice:
        _, _, fm = glo.fooof_aperiodic(fo, passes[m]["psd"][("a", c)][:, pf > 0].mean(0), SPEC_FIT["a"])
        centres += [(c, m, round(p[0], 1)) for p in fm.peak_params_ if 4.5 <= p[0] <= 9.9]
print("FOOOF peak centres in 4.5-9.9 Hz (reading a, channel-mean spectra per mouse):", len(centres), "peaks in",
      len(set((c, m) for c, m, _ in centres)), "of", 2 * len(mice), "mouse-condition spectra:", centres)

**Result: reproduced.** The mean oscillatory spectrum has a local maximum above 0 dB at 6 Hz for
both "xxxX" (3.56 dB) and "xxxx" (3.27 dB) under reading (a), and at 6 Hz under reading (b); FOOOF
finds a peak between 4.5 and 9.9 Hz in 19 of the 28 mouse and condition spectra.

## fig3ab-go-condition-spectra

> In V1, the P4 response to the unpredicted repetition "xxxX" shows a fast 2-4 dB alpha/beta increase
> that returns to baseline and long-lasting increases under 2 dB in 40-55 Hz and 80-90 Hz gamma; the
> predictable "xxxx" shows a 4-6 dB alpha/beta increase and a smaller gamma increase, both lasting
> through the first 200 ms.

In [ ]:
rows = []
for r in FIT:
    for s6 in S6:
        X, x = cond_panel(r, "go_gloexp", s6), cond_panel(r, "go_seqctl", s6)
        abX, abx = bandmean(X, *AB), bandmean(x, *AB)
        tX = np.flatnonzero(tsel(0, 0.15))[abX[tsel(0, 0.15)].argmax()]
        after = (times > times[tX]) & (times < 0.25)
        rows.append({"S3": r, "S6": s6, "xxxX_ab_at_-10ms": abX[0], "xxxx_ab_at_-10ms": abx[0],
                     "xxxX_ab_max_0_150": abX[tsel(0, 0.15)].max(), "xxxx_ab_max_0_150": abx[tsel(0, 0.15)].max(),
                     "xxxX_in_1.8_4.4": 1.8 <= abX[tsel(0, 0.15)].max() <= 4.4, "xxxx_in_3.6_6.6": 3.6 <= abx[tsel(0, 0.15)].max() <= 6.6,
                     "xxxX_ab_min_peak_to_250": abX[after].min(), "xxxX_below_1_before_250": abX[after].min() < 1,
                     "xxxx_ab_min_50_200": abx[tsel(0.05, 0.2)].min(), "xxxx_above_1_through_200": abx[tsel(0.05, 0.2)].min() > 1,
                     "xxxX_g4055_mean_150_500": X[tsel(0.15, 0.5)][:, fbins(40, 55)].mean(),
                     "xxxx_g4055_mean_150_500": x[tsel(0.15, 0.5)][:, fbins(40, 55)].mean()})
f3ab = pd.DataFrame(rows)
f3ab["xxxX_gamma_higher"] = f3ab.xxxX_g4055_mean_150_500 > f3ab.xxxx_g4055_mean_150_500
print(f3ab.round(3).T.to_string())
for r in FIT:
    for c in ["go_gloexp", "go_seqctl"]:
        show(cond_panel(r, c), f"reading {r}: {c} dB of trial-mean power over aperiodic", "frequency bin (5-150 Hz)")

**Result: not reproduced.** Read as the code draws the panels, the "xxxX" alpha/beta maximum in
the first 150 ms is in range (2.86 dB in S3 reading a, 2.51 dB in b) and falls below 1 dB before
250 ms, but the "xxxx" maximum (2.77 and 2.52 dB) is below 3.6 dB and its alpha/beta does not stay
above 1 dB through 200 ms (minimum -0.97 and -1.22 dB from 50 to 200 ms), so the time courses do
not hold. The 40-55 Hz power over 150-500 ms is higher for "xxxX" (2.07 against 1.62 dB, reading
a). The trial mean of per-trial dB values gives every alpha/beta maximum below 0 dB.

## fig3c-go-v1-contrast

> In V1 the global oddball ("xxxX" against "xxxx") gives a significant fast increase in high gamma
> (55-90 Hz) trailing into a sustained 40-55 Hz gamma increase, an immediate rise in theta power, and a
> fall in alpha/beta power over time.

In [ ]:
rows = []
TH = band("theta")
for r in FIT:
    for off in OFFS:
        p = panel_c(r, "GO", off)
        hg = p[tsel(0, 0.149)][:, fbins(55, 90) & (freqs < 90)]
        g4 = p[times > 0.3][:, fbins(40, 55)]
        th_ = p[:, TH]
        tpos = np.flatnonzero((th_ > 0).any(1))
        rows.append({"reading": r, "seed_offset": off, "hg_pos_0_150": int((hg > 0).sum()), "hg_max": hg.max(),
                     "g4055_pos_after_300": int((g4 > 0).sum()), "g4055_max_after_300": g4.max(),
                     "theta_first_pos_ms": 1000 * times[tpos[0]] if len(tpos) else np.nan, "theta_max": th_.max(),
                     "ab_neg_points": int((p[:, ABb] < 0).sum()), "ab_min": p[:, ABb].min()})
f3c = pd.DataFrame(rows)
f3c["hg_fast"] = f3c.hg_pos_0_150 > 0
f3c["g_sustained"] = f3c.g4055_pos_after_300 > 0
f3c["theta_before_150"] = f3c.theta_first_pos_ms < 150
f3c["ab_decrease"] = f3c.ab_neg_points > 0
print(f3c.round(3).to_string(index=False))
for r in FIT:
    show(panel_c(r, "GO"), f"reading {r}: GO masked contrast, channel mean", "frequency bin (5-150 Hz)")
    show(results[(r, "GO", 0)]["diff"].mean(-1), f"reading {r}: GO unmasked contrast, channel mean", "frequency bin")

**Result: reproduced.** Under both readings the masked contrast has positive 55-90 Hz values in
the first 150 ms (93 and 92 points, largest 1.05 and 0.45 dB), positive 40-55 Hz values after 300
ms (largest 0.58 and 0.25 dB), positive theta from -10 ms (largest 0.36 and 0.72 dB) and negative
alpha/beta values (down to -0.19 and -0.20 dB). The same under the five seeds.

## fig3d-go-gamma-laminar

> In V1 the global oddball raises gamma power by 0.2-0.4 dB in L2/3, L4 and L5/6 at onset,
> strengthening to about 0.6 dB by the end of the stimulus, while gamma power falls in L1.

In [ ]:
rows = []
for r in FIT:
    for bname in ["gamma", "gamma_code"]:
        for off in OFFS:
            lam = laminar(r, "GO", bname, off)
            early = lam[tsel(0, 0.099)]
            rows.append({"reading": r, "band": bname, "seed_offset": off,
                         **{f"pos_0_100_{g}": int((early[:, GROUPS[g]] > 0).sum()) for g in ["L2/3", "L4", "L5/6"]},
                         **{f"max_0_100_{g}": early[:, GROUPS[g]].max() for g in ["L2/3", "L4", "L5/6"]},
                         "max_last_100": lam[tsel(0.4, 0.5)].max(), "L1_neg_points": int((lam[:, GROUPS["L1"]] < 0).sum()),
                         "L1_min": lam[:, GROUPS["L1"]].min()})
f3d = pd.DataFrame(rows)
f3d["onset_all_three"] = (f3d[["pos_0_100_L2/3", "pos_0_100_L4", "pos_0_100_L5/6"]] > 0).all(axis=1)
f3d["late_in_0.54_0.66"] = f3d.max_last_100.between(0.54, 0.66)
f3d["L1_decrease"] = f3d.L1_neg_points > 0
f3d["outcome"] = np.where(f3d.onset_all_three & f3d["late_in_0.54_0.66"] & f3d.L1_decrease, "reproduced",
                          np.where(f3d.onset_all_three, "partially reproduced", "not reproduced"))
print(f3d.round(3).to_string(index=False))
print("\noutcome by reading (S3 fitting range, gamma band):")
print(f3d[f3d.seed_offset == 0][["reading", "band", "outcome"]].to_string(index=False))
for r in FIT:
    show(laminar(r, "GO", "gamma"), f"reading {r}: GO laminar gamma 30-90", "kept channel (top = surface)", yt=True)
lt3d = pd.concat([layer_table("GO", "gamma"), layer_table("GO", "gamma_code")])

**Result: reproduced, resting on an assumption.** Under S3 reading (a), the code's fitting range,
with the Methods' 30-90 Hz band, L2/3, L4 and L5/6 have significant positive gamma in the first
100 ms (50, 30 and 89 points), the largest value in the last 100 ms is 0.634 dB (in 0.54-0.66) and
L1 has a decrease (down to -0.51 dB). By reading (printed), this is the only one of the four that
reproduces: with the 30-50 Hz band the late value is 0.94 dB (partially reproduced), and under
reading (b) L5/6 has no early increase (0 points) with either band (not reproduced), the late
value being 0.74 dB (30-90 Hz) and 1.22 dB (30-50 Hz). The outcome therefore rests on taking the
code's fitting range with the Methods' band; the verifier of the second round rates it partially
reproduced. The onset values (0.57 to 0.75 dB) are above the paper's 0.2 to 0.4 dB, which the
criterion does not test. The L1
decrease is also in the other L1 row (start moved up by 3 rows, down to -1.00 dB).

## fig3e-go-alphabeta-laminar

> In V1 the global oddball gives an alpha/beta decrease in L1 (upper supragranular layers) and an
> alpha/beta increase starting in L5/6 that spreads up to L4 and L2/3 while weakening.

In [ ]:
f3e = pd.DataFrame([ab_summary(r, "GO", off) for r in FIT for off in OFFS])
f3e["L56_pos"] = f3e["first_pos_L5/6"].notna()
f3e["L56_first"] = f3e.L56_pos & (f3e["first_pos_L5/6"] < f3e["first_pos_L4"].fillna(np.inf)) & (f3e["first_pos_L5/6"] < f3e["first_pos_L2/3"].fillna(np.inf))
f3e["L1_neg"] = f3e["first_neg_L1"].notna()
print(f3e.round(3).to_string(index=False))
for r in FIT:
    lam = laminar(r, "GO", "alphabeta")
    print(f"reading {r}: largest positive per layer group:", {g: round(float(lam[:, GROUPS[g]].max()), 3) for g in GROUPS},
          "; most negative in L1:", round(float(lam[:, GROUPS['L1']].min()), 3))
    show(lam, f"reading {r}: GO laminar alpha/beta", "kept channel (top = surface)", yt=True)
lt3e = layer_table("GO", "alphabeta")

**Result: partially reproduced.** L1 has a significant alpha/beta decrease under both readings
(down to -0.81 and -1.00 dB) and L5/6 the largest increase (1.25 and 1.22 dB), but the first
positive bin is in L2/3 (0 and -10 ms) before L5/6 (20 and 30 ms), so the increase does not start
in L5/6. The bins 10 ms apart are within the 200 ms window's resolution.

## fig3f-go-theta-laminar

> In V1 the global oddball increases theta power, beginning in L5/6 (about 0.5 dB) within 100 ms of
> onset and spreading through the column while strengthening to 1-1.5 dB.

In [ ]:
rows = []
for r in FIT:
    for off in OFFS:
        lam = laminar(r, "GO", "theta", off)
        firsts = {g: first_time(lam, g, +1) for g in GROUPS}
        t_all = np.flatnonzero((lam > 0).any(1))
        first_layer = sorted(set(L[np.flatnonzero(lam[t_all[0]] > 0)])) if len(t_all) else []
        rows.append({"reading": r, "seed_offset": off, **{f"first_pos_{g}": v for g, v in firsts.items()},
                     "first_layers": first_layer, "max": lam.max(), "neg_points": int((lam < 0).sum())})
f3f = pd.DataFrame(rows)
f3f["L56_first_before_100"] = f3f.first_layers.map(lambda x: "L5/6" in x) & (f3f["first_pos_L5/6"] < 100)
f3f["reaches_all"] = f3f[["first_pos_L4", "first_pos_L2/3", "first_pos_L1"]].notna().all(axis=1)
f3f["max_in_0.9_1.65"] = f3f["max"].between(0.9, 1.65)
print(f3f.round(3).to_string(index=False))
for r in FIT:
    show(laminar(r, "GO", "theta"), f"reading {r}: GO laminar theta", "kept channel (top = surface)", yt=True)
lt3f = layer_table("GO", "theta")

**Result: partially reproduced.** Theta increases with no significant decrease; the first
positive bin is in L5/6 at -10 ms (alone under reading a; with L2/3 and L4 under reading b), the
largest value is 1.32 dB (reading a) and 1.44 dB (reading b), in range, but the increase never
reaches the kept L1 row, nor the other L1 row (start moved up by 3 rows).

## fig3g-go-gamma-csd-feedforward

> gamma oscillations under global oddballs in V1 show a feedforward laminar CSD pattern, starting in
> L4 and spreading toward L2/3 and L5/6.

In [ ]:
csd_table("fig3g", ff, 496)

**Result: reproduced, resting on an assumption.** With the tie-break, the L4 sink starts before
the L2/3 and L5/6 sinks under both bands (L4 -8.0, L2/3 -2.4, L5/6 4.0 ms in reading a; -12.0,
-3.2 and 6.4 ms in reading b), also at 1/16 and 1/4 of a cycle. S7's own rule ties in reading a
and in reading b puts the first L4 sink (-13.6 ms) after the others at the window start. 471 and
476 windows against 496.


## fig3h-go-alphabeta-csd-feedback

> alpha/beta oscillations in "xxxx" in V1 show a weak feedback CSD pattern, starting in L1 and
> continuing down to L2/3 and L4, with later contributions from L5/6.

In [ ]:
csd_table("fig3h", fb, 300)

**Result: not reproduced, resting on an assumption.** Recorded as reproduced before the second
verification round; the third round agreed with the change. With the sustained tie-break, under the Methods' band
(reading a) the only L1 sink that is displayed and lasts begins at 28.0 ms, after L4 (a sink from
the window start, -37.6 ms) and L2/3 (-28.8 ms). The single-sample onset at -32.0 ms that the
earlier tie-break counted is one sample, not displayed. Under the code's band (reading b) L1
(-12.0 ms) also comes after L2/3 (-20.0 ms) and L4 (-27.2 ms). The rule fails at 1/16 of a cycle
in both readings and holds at 1/4 of a cycle only in reading b. S7's own rule fails in both. 279
and 275 windows against 300.


## fig3i-go-theta-csd

> theta oscillations under global oddballs in V1 show sinks beginning in L5 and L4 and spreading to
> L2/3.

In [ ]:
G4 = ["L1", "L2/3", "L4", "L5/6"]   # the four layer groups; L5 and L6 are rows within L5/6
csd_table("fig3i", lambda f: min(f["L4"], f["L5/6"]) <= min(f[g] for g in G4) and f["L2/3"] > min(f["L4"], f["L5/6"]), 335)

**Result: not reproduced, resting on an assumption.** Recorded as reproduced before the second
verification round; the third round agreed with the change. With the sustained tie-break, under the Methods' band
(reading a) L2/3 and L4 are both sinks from the window start (-124.8 ms) once flips shorter than
1/8 of a cycle (27 samples) are merged, so the L2/3 sink does not follow the L4 sink; the earlier
-120.8 against -84.0 ms order came from brief flips. Under the code's band (reading b) L4 is a
sink from the window start and L2/3 starts at -58.4 ms, so the rule holds there. The Methods'
band is adopted. At 1/16 of a cycle reading a holds too; at 1/4 it fails. S7's own rule fails in
both readings. The outcome depends on the minimum duration, which was set for all CSD claims
together. 685 and 575 windows against 335.


## fig4ab-ssa-condition-spectra

> In V1, the response to "xxxY" shows a 2.5-5 dB alpha/beta increase at onset (returning to baseline,
> then rising again) with fast and long-lasting 2.5-5 dB increases in 40-55 Hz and 80-90 Hz gamma;
> "yyyy" shows a 2-4 dB alpha/beta increase followed by smaller (under 2 dB) gamma increases.

In [ ]:
rows = []
for r in FIT:
    for s6 in S6:
        Y, y = cond_panel(r, "lo_gloexp", s6), cond_panel(r, "igo_seqctl", s6)
        rows.append({"S3": r, "S6": s6, "xxxY_g4055_max": Y[tsel(0, 0.5)][:, fbins(40, 55)].max(),
                     "yyyy_ab_max_0_150": bandmean(y, *AB)[tsel(0, 0.15)].max(),
                     "xxxY_ab_max_0_150": bandmean(Y, *AB)[tsel(0, 0.15)].max(),
                     "yyyy_g4055_max_after_150": y[tsel(0.15, 0.5)][:, fbins(40, 55)].max()})
f4ab = pd.DataFrame(rows)
f4ab["xxxY_g_in_2.25_5.5"] = f4ab.xxxY_g4055_max.between(2.25, 5.5)
f4ab["yyyy_ab_in_1.8_4.4"] = f4ab.yyyy_ab_max_0_150.between(1.8, 4.4)
f4ab["xxxY_ab_in_2.25_5.5"] = f4ab.xxxY_ab_max_0_150.between(2.25, 5.5)
f4ab["yyyy_g_below_2.2_and_xxxY"] = (f4ab.yyyy_g4055_max_after_150 < 2.2) & (f4ab.yyyy_g4055_max_after_150 < f4ab.xxxY_g4055_max)
print(f4ab.round(3).T.to_string())
for r in FIT:
    for c in ["lo_gloexp", "igo_seqctl"]:
        show(cond_panel(r, c), f"reading {r}: {c} dB of trial-mean power over aperiodic", "frequency bin (5-150 Hz)")

**Result: partially reproduced.** Read as the code draws the panels, the "xxxY" 40-55 Hz maximum
(5.29 dB in S3 reading a, 4.53 dB in b) and the "yyyy" alpha/beta maximum (2.38 and 2.21 dB) are
in range, and the "yyyy" 40-55 Hz maximum after 150 ms is below the "xxxY" one (2.92 and 2.34 dB),
but it is above 2.2 dB, and the "xxxY" alpha/beta maximum in the first 150 ms (1.21 and 1.02 dB)
is below 2.25 dB. The ordering holds with values out of range; the trial mean of per-trial dB
values gives the same (ordering holds, every value out of range).

## fig4c-ssa-v1-contrast

> In V1 stimulus-specific adaptation ("xxxY" against "yyyy") gives a broad, non-frequency-specific
> decrease of power (0.5-1.5 dB) at onset, followed by a gamma increase (1-1.5 dB) throughout the
> stimulus.

In [ ]:
rows = []
for r in FIT:
    for off in OFFS:
        p = panel_c(r, "SSA", off)
        e = p[tsel(0, 0.15)]
        lt = p[times > 0.15][:, G]
        rows.append({"reading": r, "seed_offset": off, "early_neg_theta": int((e[:, TH] < 0).sum()), "early_neg_ab": int((e[:, ABb] < 0).sum()),
                     "early_neg_gamma": int((e[:, G] < 0).sum()), "early_min": e.min(), "late_gamma_pos": int((lt > 0).sum()), "late_gamma_max": lt.max()})
f4c = pd.DataFrame(rows)
f4c["broad_early_decrease"] = (f4c[["early_neg_theta", "early_neg_ab", "early_neg_gamma"]] > 0).all(axis=1)
f4c["early_in_0.45_1.65"] = (-f4c.early_min).between(0.45, 1.65)
f4c["early_in_legend_0.9_2.2"] = (-f4c.early_min).between(0.9, 2.2)
f4c["late_in_0.9_1.65"] = f4c.late_gamma_max.between(0.9, 1.65)
f4c["late_in_legend_0.9_2.2"] = f4c.late_gamma_max.between(0.9, 2.2)
print(f4c.round(3).to_string(index=False))
for r in FIT:
    show(panel_c(r, "SSA"), f"reading {r}: SSA masked contrast, channel mean", "frequency bin (5-150 Hz)")
    show(results[(r, "SSA", 0)]["diff"].mean(-1), f"reading {r}: SSA unmasked contrast, channel mean", "frequency bin")

**Result: reproduced.** Significant decreases in the first 150 ms span theta, alpha/beta and
gamma bins (largest -1.99 and -1.98 dB), followed by significant gamma increases after 150 ms
(largest 1.91 and 1.75 dB). Both are outside the Results' ranges and inside the legend's 1-2 dB
(0.9 to 2.2 dB), which the criterion names as its second reading. The same under the five seeds.

## fig4def-ssa-laminar

> In V1 the release from adaptation gives a fast, broad loss of gamma, alpha/beta and theta power in all
> layers, followed after about 150 ms by increases in gamma, alpha/beta and theta power that begin in L5/6
> and spread up through L4 into L2/3.

In [ ]:
rows = []
for r in FIT:
    for bname in ["gamma", "gamma_code", "alphabeta", "theta"]:
        for off in OFFS:
            lam = laminar(r, "SSA", bname, off)
            e = lam[tsel(0, 0.15)]
            late = lam.copy()
            late[times <= 0.15] = 0
            fp = {g: first_time(late, g, +1) for g in GROUPS}
            rows.append({"reading": r, "band": bname, "seed_offset": off,
                         **{f"early_neg_{g}": int((e[:, GROUPS[g]] < 0).sum()) for g in GROUPS},
                         **{f"late_first_pos_{g}": v for g, v in fp.items()}})
f4d = pd.DataFrame(rows)
f4d["early_loss_all_layers"] = (f4d[[f"early_neg_{g}" for g in GROUPS]] > 0).all(axis=1)
f4d["late_L56_first"] = f4d["late_first_pos_L5/6"].notna() & (f4d["late_first_pos_L5/6"] < f4d["late_first_pos_L4"].fillna(np.inf)) & (f4d["late_first_pos_L5/6"] < f4d["late_first_pos_L2/3"].fillna(np.inf))
f4d["band_holds"] = f4d.early_loss_all_layers & f4d.late_L56_first
print(f4d.round(1).to_string(index=False))
nb4 = {(r, gb): int(f4d[(f4d.reading == r) & (f4d.seed_offset == 0) & f4d.band.isin([gb, "alphabeta", "theta"])].band_holds.sum())
       for r in FIT for gb in ["gamma", "gamma_code"]}
print("\noutcome by reading (S3 fitting range, gamma band): bands that hold of three")
for (r, gb), n in nb4.items():
    print(f"  {r} {gb:10s} {n}  {'reproduced' if n == 3 else 'partially reproduced' if n else 'not reproduced'}")
for r in FIT:
    for bname in ["gamma", "alphabeta", "theta"]:
        show(laminar(r, "SSA", bname), f"reading {r}: SSA laminar {bname}", "kept channel (top = surface)", yt=True)
lt4 = {b: layer_table("SSA", b) for b in ["gamma", "alphabeta", "theta"]}

**Result: partially reproduced, resting on an assumption.** Under S3 reading (a) with the
Methods' 30-90 Hz gamma band, alpha/beta and theta hold (an early loss in every layer group, later
increases first in L5/6 at 160 ms, before L4 and L2/3), but in gamma the later increase starts in
L5/6 and L2/3 together (200 ms): two of three bands. With the code's 30-50 Hz band all three hold
(reproduced); under reading (b) only gamma holds, as L4 ties with L5/6 in alpha/beta and theta. By reading
(printed): partially reproduced in three of the four, reproduced with S3 (a) and 30-50 Hz.

## fig4g-ssa-gamma-csd-feedforward

> gamma oscillations in P4 of "xxxY" in V1 show a feedforward CSD pattern beginning in L4 and L5 and
> spreading into L2/3 and L6.

In [ ]:
boundary = int(np.flatnonzero(csd_rows == "L5/6")[0])
print("L4/L5 boundary row (first L5/6 CSD row):", boundary)


csd_table("fig4g", lambda f: min(f["L4"], f["L5"]) <= min(f[g] for g in G4) and f["L2/3"] > min(f["L4"], f["L5"]) and f["L6"] > min(f["L4"], f["L5"]), 1910)
for r in ["a", "b"]:
    res = CSD[("fig4g", r)]
    hit = ((res["mean"] < 0) & res["shown"])
    t_bd = res["lag"][np.flatnonzero(hit[boundary])[0]] if hit[boundary].any() else np.inf
    print(f"reading {r}: first sink in the boundary row {t_bd:.1f} ms; first L4 {res['first']['L4']:.1f}, first L5 (any) {res['first']['L5']:.1f}")

**Result: reproduced, resting on an assumption.** With the tie-break, L4 starts first under both
bands (-9.6 and -12.8 ms) and the L2/3 (-4.0 and -4.8 ms) and L6 (6.4 and 7.2 ms) sinks follow.
The rule holds at 1/16 of a cycle and with single-sample onsets, and fails at 1/4 of a cycle in
both readings. 1868 and 1943 windows against 1910.


## fig4h-ssa-alphabeta-csd

> alpha/beta oscillations in P4 of "yyyy" in V1 show sinks spreading from L5/6 and L1 into L5, L4 and
> L2/3, a feedback pattern (the Results call it mixed feedforward and feedback).

In [ ]:
csd_table("fig4h", lambda f: min(f["L1"], f["L5/6"]) < f["L4"], 319)

**Result: not reproduced, resting on an assumption.** With the tie-break, under the Methods' band
(reading a) the L4 sink (-21.6 ms) starts before the L5/6 (-12.0 ms) and L1 (16.0 ms) sinks;
under the code's band (reading b) the L1 sink (-34.4 ms) starts before L4 (25.6 ms), so the rule
holds there. The readings disagree, and the Methods' band is adopted. The same split holds at 1/16
of a cycle; at 1/4 both readings fail. With single-sample onsets both fail (reading a, L4 -36.0
against L1 -32.8 and L5/6 -28.8 ms; reading b, L4 -40.8 against L1 -40.0 and L5/6 -36.0 ms, an L1
margin of 0.8 ms). In reading a, L1 and L5/6 are sinks at the first lag while L4 is a source, so a
reading by eye could come out the other way. S7's own rule fails too. 303 and 272 windows against
319.


## fig4i-ssa-theta-csd

> theta oscillations in P4 of "xxxY" in V1 show sinks starting in L1 and L6, then activity in L2/3 at
> the trough spreading down through L4 and L5 into L6.

In [ ]:
csd_table("fig4i", lambda f: min(f["L1"], f["L6"]) <= min(f[g] for g in G4 + ["L6"]) and f["L2/3"] < f["L4"] and f["L2/3"] < f["L5"], 1266)

**Result: not reproduced, resting on an assumption.** With the tie-break, neither band has an L1
or L6 sink onset in the window; L4 (reading a) and L5 (both readings) are sinks from the window
start. The rule fails at 1/16 and 1/4 of a cycle and with single-sample onsets. S7's own rule
fails too. 2481 and 2054 windows against 1266.


## suppfig4-pupil-dd-no-difference

> Pupil size, z-scored, does not differ significantly between P4 of the random "XXXY" sequences
> and P4 of the habituation "xxxy" sequences.

Pupil area from `pupil_tracking`, blinks (`likely_blink`) set missing and filled by linear
interpolation, sampled at 30 points from P4 onset to offset (about the camera's 60 Hz). A trial
with no valid sample in the window is dropped.

In [ ]:
pt = np.linspace(0, 0.5, 30, endpoint=False)
tr = {"lo_rndctl": [], "lonaive": []}
mouse_of = {"lo_rndctl": [], "lonaive": []}
dropped = 0
for m in mice:
    eye = smeta[m][3]
    if eye is None:
        continue
    a = eye["area"].astype(float).copy()
    a[eye["blink"].astype(bool) | ~np.isfinite(a)] = np.nan
    ok = np.isfinite(a)
    for c in tr:
        for on in conds[m][c].p4_on:
            sel = (eye["t"] >= on - 0.1) & (eye["t"] <= on + 0.6) & ok
            if sel.sum() < 5:
                dropped += 1
                continue
            tr[c].append(np.interp(on + pt, eye["t"][sel], a[sel]))
            mouse_of[c].append(m)
X = {c: np.array(v) for c, v in tr.items()}
print("mice with eye tracking:", len(set(mouse_of["lo_rndctl"])), "; trials", {c: len(v) for c, v in X.items()}, "; dropped", dropped)
pooled = np.concatenate([X["lo_rndctl"], X["lonaive"]])
mu, sd = np.nanmean(pooled), np.nanstd(pooled)
Z = {c: (X[c] - mu) / sd for c in X}
Zm = {c: np.empty_like(X[c]) for c in X}
for m in set(mouse_of["lo_rndctl"]):
    both = np.concatenate([X[c][np.array(mouse_of[c]) == m] for c in X])
    for c in X:
        sel = np.array(mouse_of[c]) == m
        Zm[c][sel] = (X[c][sel] - both.mean()) / both.std()
rows = []
for name, ZZ in [("pooled z", Z), ("per-mouse z", Zm)]:
    for off in OFFS:
        f_obs, lab, pv = glo.cluster_test(ZZ["lo_rndctl"][:, :, None, None], ZZ["lonaive"][:, :, None, None],
                                          np.random.default_rng(seed("suppfig4-pupil-dd-no-difference") + off))
        d = ZZ["lo_rndctl"].mean(0) - ZZ["lonaive"].mean(0)
        rows.append({"z-scoring": name, "seed_offset": off, "clusters": len(pv), "min_p": pv.min() if len(pv) else np.nan,
                     "max_abs_mean_diff_z": np.abs(d).max(), "max_F": f_obs.max()})
print(pd.DataFrame(rows).round(4).to_string(index=False))
for name, ZZ in [("pooled z", Z), ("per-mouse z", Zm)]:
    m1, m2 = ZZ["lo_rndctl"].mean(0), ZZ["lonaive"].mean(0)
    s1 = 2 * ZZ["lo_rndctl"].std(0) / np.sqrt(len(ZZ["lo_rndctl"]))
    s2 = 2 * ZZ["lonaive"].std(0) / np.sqrt(len(ZZ["lonaive"]))
    overlap = (np.abs(m1 - m2) <= s1 + s2)
    print(f"{name}: XXXY mean z {m1.mean():.3f}, xxxy mean z {m2.mean():.3f}; 2-SEM bands overlap at {overlap.sum()} of {len(pt)} time points")

In [ ]:
# Checks asked for in verification. (1) The eye-tracking clock against the stimulus table.
# (2) A stimulus-locked pupil response: the mean z-scored pupil area over one sequence cycle (5 s,
# from 3 s before to 2 s after P4 onset) of every main-block sequence, against the same mean around as many random times in the main block
# (200 draws, the claim's second generator). (3) Time in session: XXXY against the last 50
# main-block "xxxy" of each mouse, which come just before the random block, in place of the first 50.
rng_p = np.random.default_rng(seed("suppfig4-pupil-dd-no-difference") + 1)
lag_p = np.arange(-3.0, 2.0, 0.05)
rows, late = [], {"lo_rndctl": [], "late_xxxy": []}
for m in mice:
    eye = smeta[m][3]
    if eye is None:
        continue
    s = smeta[m][1]
    a = eye["area"].astype(float).copy()
    a[eye["blink"].astype(bool) | ~np.isfinite(a)] = np.nan
    ok = np.isfinite(a) & np.isfinite(eye["t"])
    z = (a - np.nanmean(a[ok])) / np.nanstd(a[ok])
    zt, zz = eye["t"][ok], z[ok]
    main_on = s[s.kind == "main"].p4_on.to_numpy()
    locked = np.mean([np.interp(on + lag_p, zt, zz) for on in main_on], 0)
    null = []
    for _ in range(200):
        ons = rng_p.uniform(main_on.min(), main_on.max(), len(main_on))
        null.append(np.std(np.mean([np.interp(on + lag_p, zt, zz) for on in ons], 0)))
    rows.append({"mouse": m, "eye_t_first_s": np.nanmin(eye["t"]), "eye_t_last_s": np.nanmax(eye["t"]),
                 "eye_t_missing": int(np.isnan(eye["t"]).sum()), "lfp_v1_t0_s": passes[m]["t0"],
                 "stim_first_p4_s": s.p4_on.min(), "stim_last_p4_s": s.p4_on.max(),
                 "eye_dt_ms_median": 1000 * np.nanmedian(np.diff(eye["t"])),
                 "locked_sd_z": np.std(locked), "null_sd_z_95pct": np.quantile(null, 0.95),
                 "locked_above_null": np.std(locked) > np.quantile(null, 0.95),
                 "locked_min_lag_s": lag_p[locked.argmin()], "locked_max_lag_s": lag_p[locked.argmax()]})
    lo = s[(s.kind == "main") & (s.xy == "xxxy")]
    for c, ons in [("lo_rndctl", conds[m]["lo_rndctl"].p4_on), ("late_xxxy", lo.p4_on.iloc[-50:])]:
        for on in ons:
            sel = (eye["t"] >= on - 0.1) & (eye["t"] <= on + 0.6) & ok
            if sel.sum() >= 5:
                late[c].append(np.interp(on + pt, eye["t"][sel], a[sel]))
print(pd.DataFrame(rows).round(3).to_string(index=False))
XL = {c: np.array(v) for c, v in late.items()}
pl = np.concatenate([XL["lo_rndctl"], XL["late_xxxy"]])
ZL = {c: (XL[c] - np.nanmean(pl)) / np.nanstd(pl) for c in XL}
f_obs, lab, pv = glo.cluster_test(ZL["lo_rndctl"][:, :, None, None], ZL["late_xxxy"][:, :, None, None],
                                  np.random.default_rng(seed("suppfig4-pupil-dd-no-difference")))
dl = ZL["lo_rndctl"].mean(0) - ZL["late_xxxy"].mean(0)
print(f"time-in-session variant, XXXY ({len(XL['lo_rndctl'])} trials) against the last 50 main-block xxxy "
      f"({len(XL['late_xxxy'])} trials), pooled z: {len(pv)} clusters, smallest p {pv.min() if len(pv) else np.nan:.3f}; "
      f"mean z {ZL['lo_rndctl'].mean():.3f} against {ZL['late_xxxy'].mean():.3f}; largest |mean difference| {np.abs(dl).max():.3f} z")

**Result: not reproduced.** In the 9 mice with eye tracking (343 "XXXY" and 450 "xxxy" trials, 3
dropped), the cluster test over P4 finds significant differences in z-scored pupil area (smallest
cluster p 0.001 to 0.002 over the five seeds, pooled z-scoring; 0.001 per mouse), the pupil being
larger in "XXXY" (mean z 0.129 against -0.098). The eye clock is consistent with the stimulus
table: in all 9 mice the pupil averaged over the main-block sequences varies more than at random
times (SD 0.031 to 0.094 z against a 95th percentile of 0.018 to 0.038 z). The difference depends
on time in session: against the last 50 main-block "xxxy", which come just before the random
block, "XXXY" has the smaller pupil (mean z -0.104 against 0.086, cluster p 0.005).

## Summary

| Claim | Outcome |
|---|---|
| trial-counts | partially reproduced (5 of 6 counts differ by 1 to 5 trials) |
| three-contrasts-gamma-increase | reproduced |
| fig2ab-dd-condition-spectra | partially reproduced (assumption) |
| fig2c-dd-v1-contrast | partially reproduced |
| fig2d-dd-gamma-laminar | partially reproduced |
| fig2e-dd-alphabeta-laminar | partially reproduced (assumption) |
| fig2f-dd-theta | not reproduced |
| fig2g-dd-gamma-csd-feedforward | reproduced (assumption) |
| fig2h-dd-alphabeta-csd-feedback | not reproduced (assumption) |
| fig2i-dd-theta-csd-mixed | not reproduced (assumption) |
| dd-gamma-peak-frequency-v1 | reproduced |
| go-theta-peaks | reproduced |
| fig3ab-go-condition-spectra | not reproduced |
| fig3c-go-v1-contrast | reproduced |
| fig3d-go-gamma-laminar | reproduced (assumption) |
| fig3e-go-alphabeta-laminar | partially reproduced |
| fig3f-go-theta-laminar | partially reproduced |
| fig3g-go-gamma-csd-feedforward | reproduced (assumption) |
| fig3h-go-alphabeta-csd-feedback | not reproduced (assumption) |
| fig3i-go-theta-csd | not reproduced (assumption) |
| fig4ab-ssa-condition-spectra | partially reproduced |
| fig4c-ssa-v1-contrast | reproduced |
| fig4def-ssa-laminar | partially reproduced (assumption) |
| fig4g-ssa-gamma-csd-feedforward | reproduced (assumption) |
| fig4h-ssa-alphabeta-csd | not reproduced (assumption) |
| fig4i-ssa-theta-csd | not reproduced (assumption) |
| suppfig4-pupil-dd-no-difference | not reproduced |
| the ten claims across six areas | see `areas.ipynb` |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The conditions are rebuilt from the stimulus
tables, as the paper's condition file is not in the dandiset. The LFP is at 1250 Hz in the files,
so the 200 ms window is 250 samples, and the spectrogram bins lie 4.88 Hz apart (the labels 5 to
150 Hz fall on 4.9 to 151.4 Hz). Each trial is de-meaned before the spectrogram (Syncopy's
default, assumed). Three details are taken from the paper's code (epych cee2e6f): the channel
alignment and where the every-fourth downsampling starts (from the deepest row, the topmost
common row left out); the condition panels as dB of the trial-mean power over the aperiodic fit;
and, for the spectrogram fit, the code's 5-150 Hz range (S3 reading a), as the Methods' "2 to 90 Hz
with a frequency resolution of 2 Hz" fits the per-mouse spectra, not the 5 Hz spectrogram grid.
Where the S3 range changes an outcome (fig2e, fig3d, fig4def) the outcome under each reading is
printed. Where the CSD bands change an outcome the Methods' bands are adopted. S7's first-sink
rule ties at the window start, so the sink onset of each layer group's mean CSD breaks the tie,
counted for a displayed sink lasting at least 1/8 of a cycle; with that rule fig3h and fig3i do
not hold under the Methods' bands; the FIR
band-pass is an assumed windowed-sinc filter of at most 249 taps. The CSD repeats the edge rows
(Elephant's default) so that L1 has a row. The L1 results rest on one kept row; the variant with
the start moved up by 3 rows keeps the other L1 row and is printed with each L1 claim.

**What was found about the dataset.** The cited version, 0.240503.0152, has LFP timestamps that are
not in seconds for 7 of 14 mice, near-identical LFP in two pairs of mice, and one channel in four for
one mouse; version 0.240923.1441 fixes all three. The stimulus tables carry no block or condition
labels, and the habituation block holds 25 sequences where the paper describes 50 habituation
trials. The LFP is at 1250 Hz where the paper states 2.5 kHz. Three mice have no probe labelled AL
(two labelled LM twice, one RL twice) and one probe has no area label. The cohorts are 8 and 6
mice, where the paper describes two of 7, and one mouse is 88 days old, below the paper's 95 to
128. Five sessions have no eye tracking; in the other nine the eye clock is consistent with the
stimulus table.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")